# Desarrollo de sistemas de web scraping para la obtención de datos inmobiliarios y turísticos (Idealista, Fotocasa y Airbnb.)

Este cuaderno documenta, **en orden cronológico**, la evolución de los sistemas de extracción de datos desarrollados durante el Trabajo Fin de Máster. El objetivo fue construir dos bases de datos propias:

1. **Base de datos inmobiliaria** de viviendas en venta en Madrid, Barcelona y Valencia extraída de **Idealista y Fotocasa**.
2. **Base de datos turística** longitudinal de alojamientos ofertados en **Airbnb** (Madrid, Barcelona y Valencia).

El desarrollo estuvo condicionado por los mecanismos de protección frente a accesos automatizados que implementan ambas plataformas, lo que obligó a evolucionar la metodología en fases sucesivas:

| Fase | Plataforma | Estrategia | Resultado |
|------|-----------|------------|-----------|
| 1 | Idealista | Requests + BeautifulSoup | Bloqueada (HTTP 403) |
| 2 | Idealista | Selenium básico | Acceso conseguido, extracción limitada |
| 3 | Idealista | Selenium + anti-detección + simulación humana | Navegación estable |
| 4 | Idealista | Versión final: extracción estructurada completa | Primeros registros válidos |
| 5 | Airbnb | 3 ciudades, bucle diario de 365 días | Funcional pero inviable en tiempo |
| 6 | Airbnb | Optimización: muestreo temporal semanal (7 en 7) | Base de datos longitudinal |
| 7 | Airbnb | Versión final: barrios de Valencia + pausas aleatorias + reinicio de sesión | Sistema estable de larga duración |

> **Nota sobre la ejecución en Google Colab:** las fases basadas en Selenium se desarrollaron finalmente en un **entorno local (Visual Studio Code sobre Windows)**, precisamente porque Google Colab presentaba problemas de compatibilidad con navegadores gráficos (ver fase 2). Los scripts de Selenium se incluyen aquí completos como documentación del proceso; para ejecutarlos, hacerlo en local. La celda de instalación siguiente permite reproducir el entorno.

## 0. Instalación de librerías

Librerías empleadas a lo largo del proyecto: `requests` y `beautifulsoup4` (fase 1), `selenium` y `webdriver-manager` (fases 2-7) y `pandas` para la estructuración tabular y exportación a CSV en todas las fases.

In [ ]:
!pip install -q requests beautifulsoup4 pandas selenium webdriver-manager lxml

---
# 1. Sistema de extracción de datos inmobiliarios de Idealista

## 1.1. Introducción

La construcción de una base de datos propia de viviendas en venta constituyó una de las primeras fases del trabajo. Se seleccionó **Idealista** como fuente principal por su posición predominante en el mercado inmobiliario español y el elevado volumen de anuncios publicados.

La finalidad del proceso de extracción consistió en recopilar información estructurada de inmuebles en venta en Madrid: **precio, ubicación, superficie construida, número de habitaciones y descripción**.

El desarrollo estuvo condicionado por los mecanismos de protección frente a accesos automatizados de la plataforma, lo que obligó a perfeccionar distintas estrategias hasta conseguir una solución estable. La evolución se divide en tres fases principales:

1. Aproximación mediante peticiones HTTP convencionales (**Requests + BeautifulSoup**).
2. Migración a **Selenium** como herramienta de automatización web.
3. Traslado del desarrollo a un **entorno local (VS Code)** con técnicas de reducción de detección automatizada.

## 1.2. Primera aproximación: Requests, BeautifulSoup y Pandas

**Script original:** `intento webscraping con pandas.py`

La primera estrategia se fundamentó en peticiones HTTP directas para acceder a las páginas de resultados de Idealista. El planteamiento se apoyó en el análisis de la estructura de las URL: era posible acceder a distintas zonas de Madrid modificando componentes de la dirección web, lo que permitía automatizar el recorrido por los distritos (Arganzuela, Centro, Retiro, Chamberí y Salamanca).

Para aproximar las solicitudes al comportamiento de un navegador convencional se incorporaron **cabeceras HTTP personalizadas** (User-Agent, preferencias de idioma y parámetros habituales de conexión).

La lógica de extracción era:
1. Construcción dinámica de las URL de cada distrito.
2. Descarga del HTML mediante `requests`.
3. Procesamiento del código fuente con `BeautifulSoup`.
4. Identificación de anuncios mediante selectores CSS (`a.item-link`).
5. Extracción de título y enlace de cada inmueble.

In [ ]:
#intento de webscraping de idealista para obtener datos de viviendas en venta en madrid


import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

base_domain = 'https://www.idealista.com/venta-viviendas/madrid-madrid/'
url_base = 'https://www.idealista.com/venta-viviendas/madrid/'

# Viendo nuestra url base 'https://www.idealista.com/venta-viviendas/madrid/' y que para
# entrar en las diferentes zonas que nos interesan solo hay que añadir
# al final /nombre de la zona/ creo que se podria automatizar.

lista_zonas = ['arganzuela','centro','retiro','chamberi','bario-de-salamanca']

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9",
    "Accept": "text/html,application/xhtml+xml",
    "Connection": "keep-alive"
}

session = requests.Session()
session.headers.update(headers)

for zona in lista_zonas:
    url = f"{url_base}{zona}/"
    print(f"Scrapeando: {url}")

    try:
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            # buscar ttulos de anuncios
            anuncios = soup.find_all('a', class_='item-link')

            for anuncio in anuncios:
                titulo = anuncio.get_text(strip=True)
                link = base_domain + anuncio.get('href')
                print(titulo, link)

        else:
            print(f"Error en {url}: {response.status_code}")

    except Exception as e:
        print(f"Fallo en {url}: {e}")

### Problema encontrado: bloqueo HTTP 403

Las pruebas evidenciaron mecanismos de protección frente a accesos automatizados: **la mayoría de las solicitudes devolvían respuestas HTTP 403 (Forbidden)**, indicando que el servidor identificaba las peticiones como automatizadas.

Como consecuencia, el HTML obtenido no contenía la información necesaria para localizar los anuncios, imposibilitando la construcción de la base de datos. Esta limitación puso de manifiesto que **una estrategia basada exclusivamente en peticiones HTTP convencionales resultaba insuficiente** para acceder al contenido de la plataforma.

## 1.3. Migración a Selenium

**Script original:** `intento webscraping con selenium2.py`

Ante las limitaciones de la fase anterior, se optó por **Selenium** como herramienta principal de automatización. Su ventaja radica en controlar un **navegador real** y reproducir con mayor precisión el comportamiento de un usuario, lo que permite acceder a contenidos generados dinámicamente mediante JavaScript y reduce la probabilidad de detección.

La nueva arquitectura automatizaba la apertura de Google Chrome mediante **ChromeDriver** (gestionado con `webdriver-manager`), accedía a las páginas de resultados de cada distrito, gestionaba el banner de cookies y extraía título y enlace de cada anuncio, paginando con el patrón de URL `/pagina-N.htm`.

### Problemas detectados en Google Colab

Las primeras pruebas en Google Colab revelaron dificultades de compatibilidad entre Selenium y el entorno de ejecución:

- Compatibilidad limitada entre Selenium y el entorno cloud.
- Configuración compleja de ChromeDriver.
- Restricciones asociadas a la ejecución de navegadores gráficos.
- Problemas de estabilidad durante la navegación automatizada.

### Migración del desarrollo a Visual Studio Code

Para disponer de un entorno más estable, el desarrollo se trasladó a un **entorno local basado en VS Code sobre Windows**. Esta migración permitió gestionar directamente la instalación de Chrome y ChromeDriver, controlar versiones de librerías, depurar con mayor capacidad y ejecutar **procesos de larga duración** sin las restricciones de los entornos cloud gratuitos. Constituyó un punto de inflexión del proyecto.

⚠️ *Este script y los siguientes están pensados para ejecución local (las rutas de guardado apuntan a disco `D:\`).*

In [ ]:
# Intento de pelearme con Selenium para hacer webscraping de idealista y obtener datos de viviendas en venta en Madrid

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import time
import pandas as pd

url_base = 'https://www.idealista.com/venta-viviendas/madrid/'
base_domain = 'https://www.idealista.com'

lista_zonas = ['arganzuela','centro','retiro','chamberi','barrio-de-salamanca']

# Configuración para Windows - IMPORTANTE: argumentos para evitar que Chrome se cierre
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-web-resources")
options.add_argument("--no-first-run")
options.add_argument("--no-default-browser-check")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
# Agregar User-Agent para evitar bloqueos
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

service = Service(ChromeDriverManager().install())

try:
    driver = webdriver.Chrome(service=service, options=options)
    print("Conexión con Chrome establecida correctamente")
except Exception as e:
    print(f"Error al iniciar ChromeDriver: {e}")
    exit()

# Lista para almacenar los datos
datos_scraped = []

for zona in lista_zonas:
    print(f"\nScrapeando zona: {zona}")
    pagina = 1
    sin_mas_paginas = False

    while not sin_mas_paginas:
        # Construir URL según el número de página
        if pagina == 1:
            url = f"{url_base}{zona}/"
        else:
            url = f"{url_base}{zona}/pagina-{pagina}.htm"

        print(f"   Página {pagina}: {url}")

        try:
            driver.get(url)
            time.sleep(5)

            try:
                # Aceptar cookies (solo en la primera página de la primera zona)
                if pagina == 1 and zona == lista_zonas[0]:
                    try:
                        boton_cookies = driver.find_element(By.ID, "didomi-notice-agree-button")
                        boton_cookies.click()
                        time.sleep(2)
                    except:
                        print("   No se encontró botón de cookies")

                anuncios = driver.find_elements(By.CSS_SELECTOR, "a.item-link")

                # Si no hay anuncios, no hay más páginas
                if len(anuncios) == 0:
                    print(f"   No se encontraron anuncios en página {pagina}. Fin de páginas para esta zona.")
                    sin_mas_paginas = True
                else:
                    print(f"   Se encontraron {len(anuncios)} anuncios en página {pagina}")

                    for anuncio in anuncios:
                        try:
                            titulo = anuncio.text.strip()
                            link = anuncio.get_attribute("href")

                            if titulo and link:  # Validar que hay contenido
                                # Capturar datos en el diccionario
                                datos_scraped.append({
                                    'zona': zona,
                                    'pagina': pagina,
                                    'titulo': titulo,
                                    'link': link,
                                    'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
                                })
                        except Exception as e:
                            print(f"   Error al extraer anuncio: {e}")

                    pagina += 1  # Ir a la siguiente página

            except Exception as e:
                print(f"   Error procesando página {pagina}: {e}")
                sin_mas_paginas = True

        except Exception as e:
            print(f"   Error accediendo URL {url}: {e}")
            # Si falla una página, intentamos con la siguiente
            # Pero si fallan varias seguidas, abandonamos la zona
            respuestas_fallidas = 0
            respuestas_fallidas += 1
            if respuestas_fallidas > 2:
                print(f"   Demasiados errores. Abandonando zona.")
                sin_mas_paginas = True
            else:
                # Reintentar conexión si falla
                try:
                    print("   Reintentando conexión...")
                    driver.quit()
                    driver = webdriver.Chrome(service=service, options=options)
                except:
                    pass
                pagina += 1

    time.sleep(3)

driver.quit()
print("Conexión cerrada correctamente")

# Crear DataFrame
if len(datos_scraped) > 0:
    df = pd.DataFrame(datos_scraped)

    print("\n" + "="*80)
    print(f"Scraping completado. Total de anuncios: {len(df)}")
    print("="*80)
    print("Primeros registros:")
    print(df.head(10))

    # Guardar a CSV
    output_file = r'd:\Pontia\Master analisis de datos\Pruebas\datos_idealista.csv'
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Datos guardados en: {output_file}")

    # Mostrar estadísticas por zona
    print("Resumen por zona:")
    print(df['zona'].value_counts())
else:
    print("No se obtuvieron datos. Verifica que:")
    print("   1. La URL sea accesible")
    print("   2. El sitio no te esté bloqueando")
    print("   3. Los selectores CSS sean correctos")

### Limitación de esta versión

Aunque el acceso a la plataforma ya era posible, esta primera versión con Selenium solo extraía **título y enlace**, y la configuración básica del navegador seguía generando una huella de automatización detectable, provocando bloqueos intermitentes. Era necesario reforzar las medidas anti-detección y ampliar las variables extraídas.

## 1.4. Implementación de medidas de reducción de detección automatizada

**Script original:** `intento webscraping con selenium.py`

Una vez estabilizado el entorno, el principal desafío consistió en minimizar la probabilidad de detección por los mecanismos anti-bot de Idealista. La estrategia se basó en **simular el comportamiento de un usuario real**, actuando en tres frentes:

### a) Configuración avanzada del navegador
- User-Agent realista (Chrome 120 sobre Windows).
- Navegación en modo incógnito y personalización del perfil de usuario.
- Desactivación de extensiones, plugins, notificaciones e imágenes (también acelera la carga).
- Eliminación de indicadores de automatización (`excludeSwitches: enable-automation`, `useAutomationExtension: False`, `--disable-blink-features=AutomationControlled`).

### b) Modificación de propiedades JavaScript
Se sobrescriben propiedades del objeto `navigator` comúnmente usadas para detectar bots:
- `navigator.webdriver` → `undefined`
- `navigator.plugins` → lista simulada
- `navigator.languages` → `['es-ES', 'es']`

### c) Simulación de comportamiento humano y temporización (librería `time`)
- **Desplazamientos automáticos mediante scroll** (bajar a 1/4, 1/2 y volver arriba).
- Esperas tras la carga inicial de cada página.
- **Incremento progresivo de las pausas** entre páginas (`time.sleep(10 + pagina * 2)`).
- **Pausas prolongadas entre distritos** (5 minutos) para evitar patrones repetitivos.
- Detección explícita de mensajes de bloqueo, con espera larga y refresco de página si aparecen.

Durante las primeras pruebas se observó que la ejecución secuencial de solicitudes a elevada velocidad provocaba bloqueos temporales y respuestas incompletas; estas pausas temporizadas fueron determinantes para obtener los primeros registros válidos.

Además, esta versión ya incorpora la **extracción estructurada completa** (precio, habitaciones, metros cuadrados, planta y descripción) con limpieza de datos en origen: los precios se convierten a entero eliminando `€` y separadores de miles, y habitaciones/metros se extraen con expresiones regulares.

In [ ]:
# Intento de pelearme con Selenium para hacer webscraping de idealista y obtener datos de viviendas en venta en Madrid
# El programa incluye medidas avanzadas anti-detección para evitar bloqueos

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import time
import pandas as pd

url_base = 'https://www.idealista.com/venta-viviendas/madrid/'
base_domain = 'https://www.idealista.com'
lista_zonas = ['arganzuela','centro','retiro','chamberi','barrio-de-salamanca']

# Configuración para Windows - IMPORTANTE: argumentos para evitar que Chrome se cierre
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-web-resources")
options.add_argument("--no-first-run")
options.add_argument("--no-default-browser-check")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
# Agregar User-Agent más realista para evitar bloqueos
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
# Desactivar automation indicators
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# Agregar preferencias para simular navegador real
options.add_experimental_option("prefs", {
    "profile.password_manager_enabled": False,
    "credentials_enable_service": False,
    "profile.default_content_setting_values.notifications": 2,
    "profile.managed_default_content_settings.images": 2,  # Deshabilitar imágenes
    "profile.default_content_settings.popups": 0,
    "profile.default_content_settings.plugins": 2,
})
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins")
options.add_argument("--disable-images")  # Para acelerar carga
options.add_argument("--disable-background-timer-throttling")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-features=VizDisplayCompositor")
# Crear perfil temporal para evitar detección
options.add_argument("--incognito")
options.add_argument("--disable-web-security")
options.add_argument("--allow-running-insecure-content")

service = Service(ChromeDriverManager().install())

try:
    driver = webdriver.Chrome(service=service, options=options)
    print("Conexión con Chrome establecida correctamente")

    # Ejecutar JavaScript para eliminar indicadores de Selenium
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
    driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")

except Exception as e:
    print(f"Error al iniciar ChromeDriver: {e}")
    exit()

# Lista para almacenar los datos
datos_scraped = []

for zona in lista_zonas:
    print(f"\nScrapeando zona: {zona}")
    pagina = 1
    sin_mas_paginas = False
    total_anuncios_zona = 0
    anuncios_scrapeados_zona = 0

    while not sin_mas_paginas:
        # Construir URL según el número de página
        if pagina == 1:
            url = f"{url_base}{zona}/"
        else:
            url = f"{url_base}{zona}/pagina-{pagina}.htm"

        print(f"   Página {pagina}: {url}")

        try:
            driver.get(url)
            # Espera más larga para simular comportamiento humano
            time.sleep(10 + pagina * 2)  # Espera aumenta con cada página

            # Simular comportamiento humano: hacer scroll
            try:
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight/4);")
                time.sleep(2)
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
                time.sleep(1)
                driver.execute_script("window.scrollTo(0, 0);")
                time.sleep(1)
            except:
                pass  # Si falla el scroll, continuar

            try:
                # Aceptar cookies (solo en la primera página de la primera zona)
                if pagina == 1 and zona == lista_zonas[0]:
                    try:
                        boton_cookies = driver.find_element(By.ID, "didomi-notice-agree-button")
                        boton_cookies.click()
                        time.sleep(3)
                    except:
                        print("   No se encontró botón de cookies")

                # En la primera página, extraer el total de anuncios de la zona
                if pagina == 1:
                    try:
                        total_text = driver.find_element(By.ID, "h1-container__text").text.strip()
                        # Extraer el número del texto (ej: "2.342 casas y pisos en Barrio de Salamanca, Madrid")
                        import re
                        match = re.search(r'(\d+(?:\.\d+)*)', total_text)
                        if match:
                            # Remover puntos y convertir a entero
                            total_limpio = match.group(1).replace('.', '')
                            total_anuncios_zona = int(total_limpio)
                            print(f"   Total de anuncios en zona {zona}: {total_anuncios_zona}")
                        else:
                            print("   No se pudo extraer el total de anuncios")
                            total_anuncios_zona = 0
                    except Exception as e:
                        print(f"   Error al extraer total de anuncios: {e}")
                        total_anuncios_zona = 0

                # Verificar si hay mensaje de bloqueo
                try:
                    bloqueo = driver.find_element(By.XPATH, "//*[contains(text(), 'acceso no válido') or contains(text(), 'bloqueado') or contains(text(), 'detectado un uso indebido') or contains(text(), 'El acceso se ha bloqueado')]")
                    if bloqueo:
                        print(f"   ¡DETECTADO BLOQUEO! Esperando más tiempo...")
                        time.sleep(60)  # Espera muy larga si detecta bloqueo
                        # Intentar refrescar la página
                        driver.refresh()
                        time.sleep(10)
                        continue
                except:
                    pass  # No hay mensaje de bloqueo

                # Encontrar todos los contenedores de propiedades
                contenedores = driver.find_elements(By.CSS_SELECTOR, "article.item")

                # Si no hay anuncios, no hay más páginas
                if len(contenedores) == 0:
                    print(f"   No se encontraron anuncios en página {pagina}. Fin de páginas para esta zona.")
                    sin_mas_paginas = True
                else:
                    print(f"   Se encontraron {len(contenedores)} anuncios en página {pagina}")

                    for contenedor in contenedores:
                        try:
                            # Nombre/Título
                            try:
                                nombre = contenedor.find_element(By.CSS_SELECTOR, "a.item-link").text.strip()
                            except:
                                nombre = "N/A"

                            # Link
                            try:
                                link = contenedor.find_element(By.CSS_SELECTOR, "a.item-link").get_attribute("href")
                                if link and not link.startswith('http'):
                                    link = base_domain + link
                            except:
                                link = "N/A"

                            # Precio (extraer solo el número, sin símbolo € y convertir a entero)
                            try:
                                precio_text = contenedor.find_element(By.CSS_SELECTOR, "span.item-price").text.strip()
                                # Remover el símbolo €, puntos y espacios, luego convertir a entero
                                precio_limpio = precio_text.replace('€', '').replace('.', '').replace(' ', '').strip()
                                if precio_limpio.isdigit():
                                    precio = int(precio_limpio)
                                else:
                                    precio = None
                            except:
                                precio = None

                            # Extraer todos los detalles (habitaciones, metros, planta, etc.)
                            detalles = contenedor.find_elements(By.CSS_SELECTOR, "span.item-detail")

                            # Habitaciones (primer detalle) - limpiar y convertir a entero
                            try:
                                habitaciones_text = detalles[0].text.strip() if len(detalles) > 0 else ""
                                if habitaciones_text:
                                    # Extraer solo el número antes de "hab."
                                    import re
                                    match = re.search(r'(\d+)', habitaciones_text)
                                    if match:
                                        habitaciones = int(match.group(1))
                                    else:
                                        habitaciones = None
                                else:
                                    habitaciones = None
                            except:
                                habitaciones = None

                            # Metros cuadrados (segundo detalle) - limpiar y convertir a entero
                            try:
                                metros_text = detalles[1].text.strip() if len(detalles) > 1 else ""
                                if metros_text:
                                    # Extraer solo el número antes de "m²"
                                    import re
                                    match = re.search(r'(\d+)', metros_text)
                                    if match:
                                        metros_cuadrados = int(match.group(1))
                                    else:
                                        metros_cuadrados = None
                                else:
                                    metros_cuadrados = None
                            except:
                                metros_cuadrados = None

                            # Planta (tercer detalle)
                            try:
                                planta = detalles[2].text.strip() if len(detalles) > 2 else "N/A"
                            except:
                                planta = "N/A"

                            # Descripción
                            try:
                                descripcion = contenedor.find_element(By.CSS_SELECTOR, "p.ellipsis").text.strip()
                            except:
                                try:
                                    descripcion = contenedor.find_element(By.CSS_SELECTOR, "div.item-description").text.strip()
                                except:
                                    descripcion = "N/A"

                            if nombre and nombre != "N/A":  # Validar que hay contenido
                                # Capturar todos los datos en el diccionario
                                datos_scraped.append({
                                    'zona': zona,
                                    'pagina': pagina,
                                    'nombre': nombre,
                                    'precio': precio,
                                    'habitaciones': habitaciones,
                                    'metros_cuadrados': metros_cuadrados,
                                    'planta': planta,
                                    'descripcion': descripcion,
                                    'link': link,
                                })
                                anuncios_scrapeados_zona += 1
                        except Exception as e:
                            print(f"    Error al extraer anuncio: {e}")

                    # Verificar si hemos alcanzado el total de anuncios de la zona
                    if total_anuncios_zona > 0 and anuncios_scrapeados_zona >= total_anuncios_zona:
                        print(f"   ¡Completado! Se han scrapeado {anuncios_scrapeados_zona} de {total_anuncios_zona} anuncios de la zona {zona}")
                        sin_mas_paginas = True
                    else:
                        pagina += 1  # Ir a la siguiente página

            except Exception as e:
                print(f"   Error procesando página {pagina}: {e}")
                sin_mas_paginas = True

        except Exception as e:
            print(f"   Error accediendo URL {url}: {e}")
            sin_mas_paginas = True

    # Espera muy larga entre zonas para evitar detección (5 minutos)
    print(f"   Esperando 5 minutos antes de pasar a la siguiente zona...")
    time.sleep(300)
# Crear DataFrame
if len(datos_scraped) > 0:
    df = pd.DataFrame(datos_scraped)

    print("\n" + "="*80)
    print(f"Scraping completado. Total de anuncios: {len(df)}")
    print("="*80)
    print("\nColumnas del dataset:")
    print(df.columns.tolist())
    print("\nPrimeros registros:")
    print(df.head(10))

    # Guardar a CSV
    output_file = r'd:\Pontia\Master analisis de datos\Pruebas\datos_idealista.csv'
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Datos guardados en: {output_file}")

    # Mostrar estadísticas por zona
    print("Resumen por zona:")
    print(df['zona'].value_counts())
else:
    print("No se obtuvieron datos. Verifica que:")
    print("   1. La URL sea accesible")
    print("   2. El sitio no te esté bloqueando")
    print("   3. Los selectores CSS sean correctos")

## 1.5. Versión preliminar: extracción estructurada y primeros resultados válidos

**Script original:** `intento webscraping con selenium3.py`

La versión final consolidó el diseño definitivo de la estructura de extracción. Para cada anuncio se recopilan las siguientes variables:

| Variable | Tipo | Limpieza aplicada |
|----------|------|-------------------|
| `zona` | texto | distrito recorrido |
| `pagina` | entero | página de resultados |
| `nombre` | texto | título del inmueble |
| `precio` | entero | sin `€` ni puntos de miles, convertido con `int()` |
| `habitaciones` | entero | número extraído con regex |
| `metros_cuadrados` | entero | número extraído con regex |
| `planta` | texto | tercer detalle del anuncio |
| `descripcion` | texto | texto del anuncio |
| `link` | texto | URL absoluta del anuncio |

Funcionalidades destacadas del sistema final:

- **Identificación automática del número total de anuncios por distrito** (leyendo el encabezado `h1-container__text` con regex), lo que permite saber cuándo se ha completado la zona.
- Navegación secuencial entre páginas de resultados.
- Extracción estructurada de las variables de interés y almacenamiento en diccionarios → **DataFrame de Pandas**.
- **Exportación automática a CSV** (codificación UTF-8) y resumen de anuncios por zona con `value_counts()`.

La obtención de los primeros registros válidos verificó la viabilidad técnica de la metodología y sentó las bases de la base de datos inmobiliaria utilizada en los análisis posteriores.

In [ ]:
# Intento de pelearme con Selenium para hacer webscraping de idealista y obtener datos de viviendas en venta en Madrid
# El programa incluye medidas avanzadas anti-detección para evitar bloqueos

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import time
import pandas as pd

url_base = 'https://www.idealista.com/venta-viviendas/madrid/'
base_domain = 'https://www.idealista.com'
lista_zonas = ['arganzuela','centro','retiro','chamberi','barrio-de-salamanca']

# Configuración para Windows - IMPORTANTE: argumentos para evitar que Chrome se cierre
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-web-resources")
options.add_argument("--no-first-run")
options.add_argument("--no-default-browser-check")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
# Agregar User-Agent más realista para evitar bloqueos
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
# Desactivar automation indicators
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# Agregar preferencias para simular navegador real
options.add_experimental_option("prefs", {
    "profile.password_manager_enabled": False,
    "credentials_enable_service": False,
    "profile.default_content_setting_values.notifications": 2,
    "profile.managed_default_content_settings.images": 2,  # Deshabilitar imágenes
    "profile.default_content_settings.popups": 0,
    "profile.default_content_settings.plugins": 2,
})
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins")
options.add_argument("--disable-images")  # Para acelerar carga
options.add_argument("--disable-background-timer-throttling")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-features=VizDisplayCompositor")
# Crear perfil temporal para evitar detección
options.add_argument("--incognito")
options.add_argument("--disable-web-security")
options.add_argument("--allow-running-insecure-content")

service = Service(ChromeDriverManager().install())

try:
    driver = webdriver.Chrome(service=service, options=options)
    print("Conexión con Chrome establecida correctamente")

    # Ejecutar JavaScript para eliminar indicadores de Selenium
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
    driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")

except Exception as e:
    print(f"Error al iniciar ChromeDriver: {e}")
    exit()

# Lista para almacenar los datos
datos_scraped = []

for zona in lista_zonas:
    print(f"\nScrapeando zona: {zona}")
    pagina = 1
    sin_mas_paginas = False
    total_anuncios_zona = 0
    anuncios_scrapeados_zona = 0

    while not sin_mas_paginas:
        # Construir URL según el número de página
        if pagina == 1:
            url = f"{url_base}{zona}/"
        else:
            url = f"{url_base}{zona}/pagina-{pagina}.htm"

        print(f"   Página {pagina}: {url}")

        try:
            driver.get(url)
            # Espera más larga para simular comportamiento humano
            time.sleep(10 + pagina * 2)  # Espera aumenta con cada página


            try:
                # Aceptar cookies (solo en la primera página de la primera zona)
                if pagina == 1 and zona == lista_zonas[0]:
                    try:
                        boton_cookies = driver.find_element(By.ID, "didomi-notice-agree-button")
                        boton_cookies.click()
                        time.sleep(3)
                    except:
                        print("   No se encontró botón de cookies")

                # En la primera página, extraer el total de anuncios de la zona
                if pagina == 1:
                    try:
                        total_text = driver.find_element(By.ID, "h1-container__text").text.strip()
                        # Extraer el número del texto (ej: "2.342 casas y pisos en Barrio de Salamanca, Madrid")
                        import re
                        match = re.search(r'(\d+(?:\.\d+)*)', total_text)
                        if match:
                            # Remover puntos y convertir a entero
                            total_limpio = match.group(1).replace('.', '')
                            total_anuncios_zona = int(total_limpio)
                            print(f"   Total de anuncios en zona {zona}: {total_anuncios_zona}")
                        else:
                            print("   No se pudo extraer el total de anuncios")
                            total_anuncios_zona = 0
                    except Exception as e:
                        print(f"   Error al extraer total de anuncios: {e}")
                        total_anuncios_zona = 0


                # Encontrar todos los contenedores de propiedades
                contenedores = driver.find_elements(By.CSS_SELECTOR, "article.item")

                # Si no hay anuncios, no hay más páginas
                if len(contenedores) == 0:
                    print(f"   No se encontraron anuncios en página {pagina}. Fin de páginas para esta zona.")
                    sin_mas_paginas = True
                else:
                    print(f"   Se encontraron {len(contenedores)} anuncios en página {pagina}")

                    for contenedor in contenedores:
                        try:
                            # Nombre/Título
                            try:
                                nombre = contenedor.find_element(By.CSS_SELECTOR, "a.item-link").text.strip()
                            except:
                                nombre = "N/A"

                            # Link
                            try:
                                link = contenedor.find_element(By.CSS_SELECTOR, "a.item-link").get_attribute("href")
                                if link and not link.startswith('http'):
                                    link = base_domain + link
                            except:
                                link = "N/A"

                            # Precio (extraer solo el número, sin símbolo € y convertir a entero)
                            try:
                                precio_text = contenedor.find_element(By.CSS_SELECTOR, "span.item-price").text.strip()
                                # Remover el símbolo €, puntos y espacios, luego convertir a entero
                                precio_limpio = precio_text.replace('€', '').replace('.', '').replace(' ', '').strip()
                                if precio_limpio.isdigit():
                                    precio = int(precio_limpio)
                                else:
                                    precio = None
                            except:
                                precio = None

                            # Extraer todos los detalles (habitaciones, metros, planta, etc.)
                            detalles = contenedor.find_elements(By.CSS_SELECTOR, "span.item-detail")

                            # Habitaciones (primer detalle) - limpiar y convertir a entero
                            try:
                                habitaciones_text = detalles[0].text.strip() if len(detalles) > 0 else ""
                                if habitaciones_text:
                                    # Extraer solo el número antes de "hab."
                                    import re
                                    match = re.search(r'(\d+)', habitaciones_text)
                                    if match:
                                        habitaciones = int(match.group(1))
                                    else:
                                        habitaciones = None
                                else:
                                    habitaciones = None
                            except:
                                habitaciones = None

                            # Metros cuadrados (segundo detalle) - limpiar y convertir a entero
                            try:
                                metros_text = detalles[1].text.strip() if len(detalles) > 1 else ""
                                if metros_text:
                                    # Extraer solo el número antes de "m²"
                                    import re
                                    match = re.search(r'(\d+)', metros_text)
                                    if match:
                                        metros_cuadrados = int(match.group(1))
                                    else:
                                        metros_cuadrados = None
                                else:
                                    metros_cuadrados = None
                            except:
                                metros_cuadrados = None

                            # Planta (tercer detalle)
                            try:
                                planta = detalles[2].text.strip() if len(detalles) > 2 else "N/A"
                            except:
                                planta = "N/A"

                            # Descripción
                            try:
                                descripcion = contenedor.find_element(By.CSS_SELECTOR, "p.ellipsis").text.strip()
                            except:
                                try:
                                    descripcion = contenedor.find_element(By.CSS_SELECTOR, "div.item-description").text.strip()
                                except:
                                    descripcion = "N/A"

                            if nombre and nombre != "N/A":  # Validar que hay contenido
                                # Capturar todos los datos en el diccionario
                                datos_scraped.append({
                                    'zona': zona,
                                    'pagina': pagina,
                                    'nombre': nombre,
                                    'precio': precio,
                                    'habitaciones': habitaciones,
                                    'metros_cuadrados': metros_cuadrados,
                                    'planta': planta,
                                    'descripcion': descripcion,
                                    'link': link,
                                })
                                anuncios_scrapeados_zona += 1
                        except Exception as e:
                            print(f"    Error al extraer anuncio: {e}")

                    # Verificar si hemos alcanzado el total de anuncios de la zona
                    if total_anuncios_zona > 0 and anuncios_scrapeados_zona >= total_anuncios_zona:
                        print(f"   ¡Completado! Se han scrapeado {anuncios_scrapeados_zona} de {total_anuncios_zona} anuncios de la zona {zona}")
                        sin_mas_paginas = True
                    else:
                        pagina += 1  # Ir a la siguiente página

            except Exception as e:
                print(f"   Error procesando página {pagina}: {e}")
                sin_mas_paginas = True

        except Exception as e:
            print(f"   Error accediendo URL {url}: {e}")
            sin_mas_paginas = True

    # Espera muy larga entre zonas para evitar detección (5 minutos)
    print(f"   Esperando 5 minutos antes de pasar a la siguiente zona...")
    time.sleep(300)
# Crear DataFrame
if len(datos_scraped) > 0:
    df = pd.DataFrame(datos_scraped)

    print("\n" + "="*80)
    print(f"Scraping completado. Total de anuncios: {len(df)}")
    print("="*80)
    print("\nColumnas del dataset:")
    print(df.columns.tolist())
    print("\nPrimeros registros:")
    print(df.head(10))

    # Guardar a CSV
    output_file = r'd:\Pontia\Master analisis de datos\Pruebas\datos_idealista.csv'
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Datos guardados en: {output_file}")

    # Mostrar estadísticas por zona
    print("Resumen por zona:")
    print(df['zona'].value_counts())
else:
    print("No se obtuvieron datos. Verifica que:")
    print("   1. La URL sea accesible")
    print("   2. El sitio no te esté bloqueando")
    print("   3. Los selectores CSS sean correctos")

## 1.6. Versión final: Corrección de errores y mejora en estructura de extracción de contenedores

**Script original:** `Web scraping Idealista corregido y mejorado Barcelona (GSH).py`

Tomando como base la versión preliminar anterior, se revisó en detalle el CSV resultante y se detectó un error sistemático en la extracción de las características de los anuncios: **cuando un inmueble incluye plaza de garaje**, Idealista añade un elemento adicional al principio de la lista de detalles (`span.item-detail`) con un texto como *"Garaje incluido"*. El código original asumía siempre una posición fija para habitaciones, metros y planta, por lo que en esos anuncios los tres campos quedaban desplazados una posición y mal encuadrados, sin que saltara ningún error visible.

La corrección consistió en detectar previamente si algún elemento de la lista contiene la palabra "garaje" y, en función de ello, calcular un `offset` (0 o 1) que se suma al índice de lectura de habitaciones, metros y planta. De este modo el encuadre es correcto tanto si el anuncio tiene garaje como si no.

Adicionalmente, se incorporaron dos mejoras de robustez ya utilizadas en el desarrollo de Fotocasa (ver sección 2):

- **Guardado incremental por zona**: el CSV se sobrescribe con los datos acumulados al terminar cada zona, en lugar de solo al finalizar todo el proceso, evitando pérdidas de información ante cualquier interrupción.
- **Límite de páginas configurable** (`if pagina == 999: break`) para poder acotar la ejecución durante las pruebas.

*La lista `lista_zonas` de la celda siguiente incluye solo dos distritos de Barcelona a modo de ejemplo. El mismo script se ejecutó de forma iterativa cambiando `lista_zonas` y `url_base` para cubrir todos los distritos de Madrid, Barcelona y Valencia, generando un CSV por tanda que posteriormente se combina en la sección 5.*

In [ ]:

# Codigo para el scrap de idealista Barcelona EDITADO Y ADAPTADO POR GERMÁN
# v2: añadido guardado incremental después de cada zona (mismo patrón que Fotocasa v6)


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import time
import re
import os
import pandas as pd

url_base    = 'https://www.idealista.com/venta-viviendas/barcelona/'
base_domain = 'https://www.idealista.com'
lista_zonas = ['eixample', 'gracia'] # Ir añadiendo en función de la necesidad: 'eixample', 'gracia', 'sant-marti', 'sarria-sant-gervasi', 'les-corts', 'sants-montjuic', 'horta-guinardo', 'nou-barris', 'sant-andreu', 'ciutat-vella'

# ── RUTA DE SALIDA ───────────────────────────────────────────
# Movida al principio (igual que en Fotocasa v6) --> guardado incremental dentro del bucle
# Algo que he descubierto trabajando el código de Fotocasa, es que es mucho más útil el añadir un recurso para que se vayan guardando los datos en csv según el bot avanza entre páginas.
# Cuando el bot extrae los datos de una nueva página, sobreescribe el csv generado al principio para ir añadiendo datos.

output_file = r"C:\Users\German\Desktop\Data Analysis Resources\Proyecto Jupiter\Código\Resultados scrap\Scraps ya hechos Idealista\datos_idealista_barcelona_final.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Configuración para Windows - IMPORTANTE: argumentos para evitar que Chrome se cierre
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-web-resources")
options.add_argument("--no-first-run")
options.add_argument("--no-default-browser-check")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
# Agregar User-Agent más realista para evitar bloqueos
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
# Desactivar automation indicators
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# Agregar preferencias para simular navegador real
options.add_experimental_option("prefs", {
    "profile.password_manager_enabled": False,
    "credentials_enable_service": False,
    "profile.default_content_setting_values.notifications": 2,
    "profile.managed_default_content_settings.images": 2,  # Deshabilitar imágenes
    "profile.default_content_settings.popups": 0,
    "profile.default_content_settings.plugins": 2,
})
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins")
options.add_argument("--disable-images")  # Para acelerar carga
options.add_argument("--disable-background-timer-throttling")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-features=VizDisplayCompositor")
# Crear perfil temporal para evitar detección
options.add_argument("--incognito")
options.add_argument("--disable-web-security")
options.add_argument("--allow-running-insecure-content")

service = Service(ChromeDriverManager().install())

try:
    driver = webdriver.Chrome(service=service, options=options)
    print("Conexión con Chrome establecida correctamente")

    # Ejecutar JavaScript para eliminar indicadores de Selenium
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
    driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")

except Exception as e:
    print(f"Error al iniciar ChromeDriver: {e}")
    exit()

# Lista para almacenar los datos
datos_scraped = []

for zona in lista_zonas:
    print(f"\nScrapeando zona: {zona}")
    pagina = 1
    sin_mas_paginas = False
    total_anuncios_zona = 0
    anuncios_scrapeados_zona = 0

    while not sin_mas_paginas:
        # Construir URL según el número de página
        if pagina == 1:
            url = f"{url_base}{zona}/"
        else:
            url = f"{url_base}{zona}/pagina-{pagina}.htm"

        print(f"   Página {pagina}: {url}")

        try:
            driver.get(url)
            # Espera más larga para simular comportamiento humano
            time.sleep(10 + pagina * 2)  # Espera aumenta con cada página


            try:
                # Aceptar cookies (solo en la primera página de la primera zona)
                if pagina == 1 and zona == lista_zonas[0]:
                    try:
                        boton_cookies = driver.find_element(By.ID, "didomi-notice-agree-button")
                        boton_cookies.click()
                        time.sleep(3)
                    except:
                        print("   No se encontró botón de cookies")

                # En la primera página, extraer el total de anuncios de la zona
                if pagina == 1:
                    try:
                        total_text = driver.find_element(By.ID, "h1-container__text").text.strip()
                        # Extraer el número del texto (ej: "2.342 casas y pisos en Barrio de Salamanca, Madrid")
                        match = re.search(r'(\d+(?:\.\d+)*)', total_text)
                        if match:
                            # Remover puntos y convertir a entero
                            total_limpio = match.group(1).replace('.', '')
                            total_anuncios_zona = int(total_limpio)
                            print(f"   Total de anuncios en zona {zona}: {total_anuncios_zona}")
                        else:
                            print("   No se pudo extraer el total de anuncios")
                            total_anuncios_zona = 0
                    except Exception as e:
                        print(f"   Error al extraer total de anuncios: {e}")
                        total_anuncios_zona = 0


                # Encontrar todos los contenedores de propiedades
                contenedores = driver.find_elements(By.CSS_SELECTOR, "article.item")

                # Si no hay anuncios, no hay más páginas
                if len(contenedores) == 0:
                    print(f"   No se encontraron anuncios en página {pagina}. Fin de páginas para esta zona.")
                    sin_mas_paginas = True
                else:
                    print(f"   Se encontraron {len(contenedores)} anuncios en página {pagina}")

                    for contenedor in contenedores:
                        try:
                            # Nombre/Título
                            try:
                                nombre = contenedor.find_element(By.CSS_SELECTOR, "a.item-link").text.strip()
                            except:
                                nombre = "N/A"

                            # Link
                            try:
                                link = contenedor.find_element(By.CSS_SELECTOR, "a.item-link").get_attribute("href")
                                if link and not link.startswith('http'):
                                    link = base_domain + link
                            except:
                                link = "N/A"

                            # Precio (extraer solo el número, sin símbolo € y convertir a entero)
                            try:
                                precio_text = contenedor.find_element(By.CSS_SELECTOR, "span.item-price").text.strip()
                                # Remover el símbolo €, puntos y espacios, luego convertir a entero
                                precio_limpio = precio_text.replace('€', '').replace('.', '').replace(' ', '').strip()
                                if precio_limpio.isdigit():
                                    precio = int(precio_limpio)
                                else:
                                    precio = None
                            except:
                                precio = None

                            # Extraer todos los detalles (habitaciones, metros, planta, etc.)
                            detalles = contenedor.find_elements(By.CSS_SELECTOR, "span.item-detail")
                            # Detectar si hay garaje en los detalles
                            tiene_garaje = any('garaje' in d.text.lower() for d in detalles)
                            # Para que el índice cambie en función de si hay garaje, usamos un "offset" para que cambien los índices
                            offset = 0

                            # Ahora ya metemos Garaje y el resto en función del offset:
                            try:
                                if tiene_garaje:
                                    garaje = detalles[0].text.strip()
                                    offset = 1
                                else:
                                    garaje = "No"

                            except:
                                garaje = "N/A"

                            # Habitaciones (primer detalle) - limpiar y convertir a entero
                            try:
                                habitaciones_text = detalles[offset].text.strip() if len(detalles) > offset else ""
                                if habitaciones_text:
                                    # Extraer solo el número antes de "hab."
                                    match = re.search(r'(\d+)', habitaciones_text)
                                    if match:
                                        habitaciones = int(match.group(1))
                                    else:
                                        habitaciones = None
                                else:
                                    habitaciones = None
                            except:
                                habitaciones = None

                            # Metros cuadrados (segundo detalle) - limpiar y convertir a entero
                            try:
                                metros_text = detalles[offset + 1].text.strip() if len(detalles) > offset + 1 else ""
                                if metros_text:
                                    # Extraer solo el número antes de "m²"
                                    match = re.search(r'(\d+)', metros_text)
                                    if match:
                                        metros_cuadrados = int(match.group(1))
                                    else:
                                        metros_cuadrados = None
                                else:
                                    metros_cuadrados = None
                            except:
                                metros_cuadrados = None

                            # Planta (tercer detalle)
                            try:
                                planta = detalles[offset + 2].text.strip() if len(detalles) > offset + 1 else "N/A"
                            except:
                                planta = "N/A"

                            # Descripción
                            try:
                                descripcion = contenedor.find_element(By.CSS_SELECTOR, "p.ellipsis").text.strip()
                            except:
                                try:
                                    descripcion = contenedor.find_element(By.CSS_SELECTOR, "div.item-description").text.strip()
                                except:
                                    descripcion = "N/A"

                            if nombre and nombre != "N/A":  # Validar que hay contenido
                                # Capturar todos los datos en el diccionario
                                datos_scraped.append({
                                    'zona': zona,
                                    'pagina': pagina,
                                    'nombre': nombre,
                                    'precio': precio,
                                    'garaje': garaje,
                                    'habitaciones': habitaciones,
                                    'metros_cuadrados': metros_cuadrados,
                                    'planta': planta,
                                    'descripcion': descripcion,
                                    'link': link,
                                })
                                anuncios_scrapeados_zona += 1
                        except Exception as e:
                            print(f"    Error al extraer anuncio: {e}")

                    # Verificar si hemos alcanzado el total de anuncios de la zona
                    if total_anuncios_zona > 0 and anuncios_scrapeados_zona >= total_anuncios_zona:
                        print(f"   ¡Completado! Se han scrapeado {anuncios_scrapeados_zona} de {total_anuncios_zona} anuncios de la zona {zona}")
                        sin_mas_paginas = True
                    else:
                        pagina += 1  # Ir a la siguiente página

        # CLAVE PARA PRUEBAS --- para ir haciendo pruebas, limito a solo unas pocas páginas
                    if pagina == 999:
                        print(f"Corte de prueba alcanzado ({pagina} páginas).")
                        break

            except Exception as e:
                print(f"   Error procesando página {pagina}: {e}")
                sin_mas_paginas = True

        except Exception as e:
            print(f"   Error accediendo URL {url}: {e}")
            sin_mas_paginas = True

    # ── GUARDADO INCREMENTAL después de cada zona ────────────
    # Mismo patrón que Fotocasa v6: si el proceso se interrumpe,
    # los datos de las zonas ya completadas no se pierden
    if len(datos_scraped) > 0:
        df_parcial = pd.DataFrame(datos_scraped)
        df_parcial.to_csv(output_file, index=False, encoding='utf-8')
        print(f"   CSV actualizado: {len(datos_scraped)} registros totales guardados.")

    # Espera muy larga entre zonas para evitar detección (5 minutos)
    print(f"   Esperando 5 minutos antes de pasar a la siguiente zona...")
    time.sleep(300)

# ── RESUMEN FINAL ────────────────────────────────────────────
if len(datos_scraped) > 0:
    df = pd.DataFrame(datos_scraped)

    print("\n" + "="*80)
    print(f"Scraping completado. Total de anuncios: {len(df)}")
    print("="*80)
    print("\nColumnas del dataset:")
    print(df.columns.tolist())
    print("\nPrimeros registros:")
    print(df.head(10))

    # Guardado final (redundante pero seguro)
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Datos guardados en: {output_file}")

    # Mostrar estadísticas por zona
    print("Resumen por zona:")
    print(df['zona'].value_counts())
else:
    print("No se obtuvieron datos. Verifica que:")
    print("   1. La URL sea accesible")
    print("   2. El sitio no te esté bloqueando")
    print("   3. Los selectores CSS sean correctos")

---
# 2. Sistema de extracción de datos inmobiliarios de Fotocasa

## 2.1. Introducción

Para complementar la base de datos de Idealista y ampliar la cobertura del mercado de compraventa, se incorporó **Fotocasa** como segunda fuente de anuncios inmobiliarios. El objetivo de variables a extraer es equivalente al de Idealista (precio, ubicación, superficie, habitaciones, planta y descripción), añadiendo además el **barrio** del inmueble y el **link** del anuncio como identificador único, imprescindible para poder deduplicar y combinar los datos de ambos portales en la sección 5.

Aunque la lógica general de recorrido por zonas y paginación se heredó del scraper de Idealista, Fotocasa presenta una estructura HTML completamente distinta —al usar un framework de frontend diferente— y un sistema de detección de bots notablemente más agresivo, lo que obligó a un desarrollo iterativo propio.

## 2.2. Comparativa de estructura HTML entre portales

El primer intento de adaptación, reutilizando literalmente los selectores CSS de Idealista (`a.item-link`, `span.item-price`, `span.item-detail`), no encontraba ningún dato en el HTML de Fotocasa. Fue necesario comparar campo a campo la estructura de un anuncio en ambos portales:

| Campo | Idealista | Fotocasa |
|---|---|---|
| Contenedor del anuncio | `article.item` | `article` (clase dinámica `@container` no utilizable como selector) |
| Nombre / título | `a.item-link` con texto directo | `<h3><a><span>` anidado |
| Precio | `span.item-price` | `div.text-display-3 span`, sin clase específica reutilizable |
| Barrio | No existe campo equivalente | `p.text-body-1` |
| Descripción | `p.ellipsis` / `div.item-description` | `p.text-body-2`, dentro de un collapsible "Más detalles" cerrado por defecto (el texto ya está en el DOM sin necesidad de expandirlo) |
| Características | Varios `span.item-detail` independientes | Una única `ul.text-body-1` con varios `li` |

## 2.3. El bloqueo en la página 2 y la estrategia de sesión por página

Una vez adaptados los selectores, el scraper extraía correctamente los anuncios de la primera página de cualquier búsqueda, pero **Fotocasa bloqueaba sistemáticamente el acceso a la página 2**, devolviendo una página de error ("Sentimos la interrupción"). Se probaron sin éxito definitivo varias estrategias habituales: esperas más largas y aleatorias, scroll progresivo simulando lectura humana, y retirar de la configuración de Chrome los argumentos `--disable-images` e `--incognito` (ambos, señales reconocibles de automatización para los sistemas de detección modernos).

La solución que finalmente resultó efectiva fue **abrir y cerrar una sesión nueva de Chrome para cada página** en lugar de mantener una única sesión abierta durante todo el recorrido de un distrito, eliminando así el historial de sesión que Fotocasa parecía estar utilizando para detectar el patrón automatizado. El código se organizó en tres funciones:

- `crear_driver()` — centraliza la configuración de Chrome y las medidas anti-detección, invocada una vez por página.
- `extraer_detalles()` — parsea la lista de características (`ul.text-body-1`) y separa habitaciones, baños, metros, planta y extras.
- `scrapear_pagina()` — encapsula el ciclo completo de una página (apertura de sesión, cookies, comprobación de página de error con reintento automático, scroll, extracción y cierre garantizado del driver mediante `finally`).

Al igual que en Idealista, se incorporó guardado incremental del CSV tras cada página, para no perder progreso ante una interrupción en procesos de varias horas de duración.

*La lista `lista_barrios` de la celda siguiente incluye solo dos distritos de Barcelona a modo de ejemplo. El mismo script se ejecutó de forma iterativa cambiando `lista_barrios` y `url_base` para cubrir todos los distritos de Madrid, Barcelona y Valencia.*

In [ ]:
# WEBSCRAPING FOTOCASA - VERSIÓN FINAL
# Editado y adaptado por Germán | Corregido por Claude
# ============================================================

# ── LIBRERÍAS ────────────────────────────────────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import time
import re
import os
import pandas as pd
import random as rd

# ── CONFIGURACIÓN ────────────────────────────────────────────
# Aquí meto la url general desde la que construir las de cada página y cada zona,
# y a su vez la lista de zonas (en este caso distritos pero lo llamo barrios)
url_base     = 'https://www.fotocasa.es/es/comprar/viviendas/barcelona-capital/' # cambiar la url base a las de madrid y valencia cuando toque
lista_barrios = ['eixample', 'gracia']  # Ampliar con más distritos cuando proceda

# Ruta de salida para el CSV
output_file = (
    r"C:\Users\German\Desktop\Data Analysis Resources"
    r"\Proyecto Jupiter\Código\Resultados scrap\datos_fotocasa.csv"
)

# Límite de páginas por distrito para pruebas
# → Cambiar a un número grande (ej: 999) para el scraping completo
LIMITE_PAGINAS = 35


# ── FUNCIÓN: crear driver ─────────────────────────────────────
# Centralizo aquí la creación del driver para no repetir 30 líneas
# en cada iteración. Se llama una vez por página.
def crear_driver():
    # Lo primero es añadir todos los recursos (options) que harán que sea posible
    # acceder a la página, así como que la página no nos detecte al entrar
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-web-resources")
    options.add_argument("--no-first-run")
    options.add_argument("--no-default-browser-check")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_experimental_option("prefs", {
        "profile.password_manager_enabled": False,
        "credentials_enable_service": False,
        "profile.default_content_setting_values.notifications": 2,
        # Imágenes activadas: desactivarlas activa la detección de bot en Fotocasa
        "profile.default_content_settings.popups": 0,
        "profile.default_content_settings.plugins": 2,
    })
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-plugins")
    # Sin --disable-images ni --incognito: ambos son señales de bot para Fotocasa
    options.add_argument("--disable-background-timer-throttling")
    options.add_argument("--disable-backgrounding-occluded-windows")
    options.add_argument("--disable-renderer-backgrounding")
    options.add_argument("--disable-features=VizDisplayCompositor")
    options.add_argument("--disable-web-security")
    options.add_argument("--allow-running-insecure-content")

    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=options)

    # Eliminar indicadores de Selenium del navigator
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
    driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")

    return driver


# ── FUNCIÓN AUXILIAR: extraer detalles ───────────────────────
# Al contrario que en IDEALISTA, me resulta más fácil o me ayuda a simplificar
# el definir una función con la que extraer detalles de la lista de
# características, ya que en Fotocasa están en la misma línea.
# Def → Extrae los items de la lista de características del anuncio
# (habitaciones, baños, m², planta, ascensor, etc.).
# Devuelve un diccionario con campos separados para facilitar el análisis posterior.
def extraer_detalles(contenedor):
    habitaciones   = "N/A"
    banos          = "N/A"
    metros         = "N/A"
    planta         = "N/A"
    extras         = []     # Lista con el resto (Ascensor, Calefacción, AC…)
                            # será más fácil trabajar luego en el df y Colab
    detalles_crudo = "N/A"  # String completo por si acaso

    try:
        # La línea donde están los detalles clave que buscamos
        linea_detalles = contenedor.find_element(By.CSS_SELECTOR, "ul.text-body-1")
        items          = linea_detalles.find_elements(By.TAG_NAME, "li")
        textos         = [li.text.strip() for li in items if li.text.strip()]
        detalles_crudo = " · ".join(textos)

        for t in textos:
            t_lower = t.lower()
            if "hab" in t_lower:
                habitaciones = t
            elif "baño" in t_lower:
                banos = t
            elif "m²" in t_lower or "m2" in t_lower:
                metros = t
            elif "planta" in t_lower or "entresuelo" in t_lower or "bajos" in t_lower:
                planta = t
            else:
                extras.append(t)

    except Exception:
        pass

    return {
        "habitaciones":   habitaciones,
        "banos":          banos,
        "metros":         metros,
        "planta":         planta,
        "extras":         " | ".join(extras) if extras else "N/A",
        "detalles_crudo": detalles_crudo,
    }


# ── FUNCIÓN: scrapear una página concreta ───────────────────
# Toda la lógica de entrar a una URL, esperar, scrollear y extraer
# artículos queda encapsulada aquí. Devuelve lista de dicts con los datos.
def scrapear_pagina(url, es_primera_pagina):

    print(f"\n   Abriendo nueva sesión de Chrome para: {url}")
    driver = None

    try:
        driver = crear_driver()
        driver.get(url)

        # ── Espera inicial ────────────────────────────────
        # Primera página espera más porque tiene el banner de cookies... pero al cambiar el bucle a que entre y salga, reduzco este tiempo
        espera_base = 5 if es_primera_pagina else 12
        time.sleep(espera_base + rd.uniform(1, 4))

        # ── Aceptar cookies ───────────────────────────────
        # En cada sesión nueva el banner vuelve a aparecer, así que hay que aceptarlo siempre
        try:
            boton_cookies = driver.find_element(By.ID, "didomi-notice-agree-button")
            boton_cookies.click()
            print("   Cookies aceptadas correctamente")
            time.sleep(rd.uniform(4, 7))
        except Exception:
            print("   No se encontró botón de cookies")

        # ── Comprobar que no estamos en página de error ───
        titulo_pagina = driver.title.lower()
        if "sentimos" in titulo_pagina or "error" in titulo_pagina:
            print(f"  Fotocasa devolvió página de error en la carga inicial.")
            print(f"   Esperando 90s y reintentando una vez...")
            time.sleep(90)
            driver.get(url)
            time.sleep(rd.uniform(16, 22))
            titulo_pagina = driver.title.lower()
            if "sentimos" in titulo_pagina or "error" in titulo_pagina:
                print(f"  Sigue en error tras reintento. Saltando esta página.")
                return [], 0  # Devuelve lista vacía y 0 como total

        # ── Extraer total de anuncios (solo en primera página de cada distrito) ──
        total_anuncios = 0
        if es_primera_pagina:
            try:
                total_text = driver.find_element(By.TAG_NAME, "h1").text.strip()
                print(f"   Encabezado detectado: {total_text}")
                match = re.search(r'(\d+(?:\.\d+)*)', total_text)
                if match:
                    total_limpio   = match.group(1).replace('.', '')
                    total_anuncios = int(total_limpio)
                    print(f"   Total de anuncios en este distrito: {total_anuncios}")
                else:
                    print("   No se pudo extraer el total de anuncios del H1")
            except Exception as e:
                print(f"   Error al extraer total de anuncios: {e}")

        # ── Scroll para forzar carga de todos los artículos ──
        # Bajamos despacio simulando lectura humana y luego subimos
        # La opción más adecuada es usar random.uniform para que sea aleatorio
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.25);")
        time.sleep(rd.uniform(1.5, 3))
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.5);")
        time.sleep(rd.uniform(1.5, 3))
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.75);")
        time.sleep(rd.uniform(1.5, 3))
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(rd.uniform(2, 4))
        driver.execute_script("window.scrollTo(0, 0);")  # Con esto volvemos a subir
        time.sleep(rd.uniform(1, 2))

        # ── Encontrar contenedores (artículos) ──────────
        contenedores = driver.find_elements(By.CSS_SELECTOR, "article")
        print(f"   Artículos encontrados en el DOM: {len(contenedores)}")

        if len(contenedores) == 0:
            print("   No se encontraron artículos. Posible fin de páginas o bloqueo.")
            return [], total_anuncios

        # ── Extraer datos de cada artículo ───────────────
        datos_pagina = []

        for contenedor in contenedores:
            try:
                # Parece que lee algunos anuncios (banners) como contenedores y por eso
                # me saca tantos anuncios con todo "N/A". Los saltamos con esto.
                texto_h3 = ""
                try:
                    texto_h3 = contenedor.find_element(By.CSS_SELECTOR, "h3").text.strip()
                except Exception:
                    pass
                if not texto_h3 or len(texto_h3) < 5:
                    continue  # Saltar este contenedor, es un banner publicitario

                # ── PRECIO ───────────────────────────────────────
                # Esta parte ha sido muy complicada porque la clave de html
                # del precio cambia entre anuncios
                precio = "N/A"
                try:
                    # El precio está en un <span> dentro del div con clase text-display-3
                    precio_elem = contenedor.find_element(
                        By.CSS_SELECTOR, "div.text-display-3 span"
                    )
                    precio = precio_elem.text.strip()
                except Exception:
                    pass
                # Fallback 1: buscar cualquier span que contenga € en el contenedor
                if precio == "N/A":
                    try:
                        spans = contenedor.find_elements(By.TAG_NAME, "span")
                        for span in spans:
                            texto_span = span.text.strip()
                            if "€" in texto_span and len(texto_span) < 20:
                                precio = texto_span
                                break
                    except Exception:
                        pass
                # Fallback 2: xpath buscando directamente el texto con €
                if precio == "N/A":
                    try:
                        elems = contenedor.find_elements(
                            By.XPATH, ".//span[contains(text(),'€')]"
                        )
                        for elem in elems:
                            texto_elem = elem.text.strip()
                            if len(texto_elem) < 25:
                                precio = texto_elem
                                break
                    except Exception:
                        pass

                # ── PUBLICACIÓN (antigüedad del anuncio) ─────────
                publicacion = "N/A"
                try:
                    # La ul con text-small contiene: antigüedad + badge (Top+, etc.)
                    # El primer <li> es siempre la antigüedad
                    ul_meta  = contenedor.find_element(By.CSS_SELECTOR, "ul.text-small")
                    li_items = ul_meta.find_elements(By.TAG_NAME, "li")
                    if li_items:
                        publicacion = li_items[0].text.strip()
                except Exception:
                    pass

                # ── NOMBRE DEL ANUNCIO ────────────────────────────
                nombre = "N/A"
                try:
                    # El nombre está en <h3><a><span>...</span></a></h3>
                    # Pero no en todos los casos, así que hay que meter
                    # otra condición si el nombre es N/A
                    nombre_elem = contenedor.find_element(By.CSS_SELECTOR, "h3 a span")
                    nombre      = nombre_elem.text.strip()
                except Exception:
                    pass
                if nombre == "N/A":
                    try:
                        nombre = contenedor.find_element(By.CSS_SELECTOR, "h3").text.strip()
                    except Exception:
                        pass

                # ── LINK DEL ANUNCIO ─────────────────────────────────────
                link = "N/A"
                try:
                     link_elem = contenedor.find_element(By.CSS_SELECTOR, "h3 a")
                     href = link_elem.get_attribute("href")
                     # href ya viene como URL absoluta desde Selenium (a diferencia de BeautifulSoup)
                     if href:
                         link = href
                except Exception:
                    pass

                # ── BARRIO / UBICACIÓN ────────────────────────────
                barrio_anuncio = "N/A"
                try:
                    # <p class="text-body-1 text-on-surface opacity-75 truncate ...">
                    # Tiene varias clases; buscamos la que incluye text-body-1
                    barrio_elem    = contenedor.find_element(By.CSS_SELECTOR, "p.text-body-1")
                    barrio_anuncio = barrio_elem.text.strip()
                except Exception:
                    pass

                # ── DETALLES (hab, baños, m², planta, extras) ─────
                detalles = extraer_detalles(contenedor)

                # ── DESCRIPCIÓN ───────────────────────────────────
                # La descripción está en un collapsible. El texto ya está en el DOM
                # aunque el collapsible esté visualmente cerrado, así que lo leemos
                # directamente sin hacer click (los clicks causaban detección de bot)
                descripcion = ""
                try:
                    desc_elem   = contenedor.find_element(By.CSS_SELECTOR, "p.text-body-2")
                    descripcion = desc_elem.text.strip()
                except Exception:
                    pass

                # ── Validación mínima y guardado ─────────────────
                # Para hacer la validación mínima y que tenga sentido el guardado,
                # solo descartamos si no hay ningún campo clave.
                # Si esto se cumple, entonces usamos .append() para meter en cada
                # clave del diccionario el valor scrapeado
                if barrio_anuncio != "N/A" or nombre != "N/A" or precio != "N/A":
                    datos_pagina.append({
                        'precio':          precio,
                        'publicacion':     publicacion,
                        'nombre':          nombre,
                        'link' :           link,
                        'barrio':          barrio_anuncio,
                        'habitaciones':    detalles['habitaciones'],
                        'banos':           detalles['banos'],
                        'metros':          detalles['metros'],
                        'planta':          detalles['planta'],
                        'extras':          detalles['extras'],
                        'detalles_crudo':  detalles['detalles_crudo'],
                        'descripcion':     descripcion,
                    })

            except Exception as e:
                print(f"      Error al extraer anuncio: {e}")

        print(f"   Anuncios válidos extraídos de esta página: {len(datos_pagina)}")
        return datos_pagina, total_anuncios

    except Exception as e:
        print(f"   Error general en scrapear_pagina: {e}")
        return [], 0

    finally:
        # IMPORTANTE: siempre cerramos el driver al terminar la página,
        # tanto si fue bien como si hubo error. Esto es lo que elimina
        # el historial de sesión que Fotocasa usaba para detectarnos.
        if driver:
            try:
                driver.quit()
                print("   Sesión de Chrome cerrada correctamente.")
            except Exception:
                pass


# ── SCRAPING PRINCIPAL ───────────────────────────────────────
# Una vez que hemos definido toda la función para extraer los campos de la
# línea de detalles, pasamos a definir el scrapeo en general de forma
# parecida a con Idealista, pero con la nueva estrategia de sesión por página
datos_scraped = []

# Asegurar que existe la carpeta de salida antes de empezar
os.makedirs(os.path.dirname(output_file), exist_ok=True)

for barrio in lista_barrios:
    print(f"\n{'='*60}")
    print(f"Scrapeando distrito: {barrio}")
    print(f"{'='*60}")

    pagina                    = 1
    total_anuncios_barrio     = 0
    anuncios_scrapeados_barrio = 0

    while True:

        # ── Construir URL ────────────────────────────────────
        if pagina == 1:
            url = f"{url_base}{barrio}/l"
        else:
            url = f"{url_base}{barrio}/l/{pagina}"

        print(f"\n   Página {pagina}: {url}")

        # ── Scrapear esta página en sesión nueva ─────────────
        es_primera = (pagina == 1)
        datos_pagina, total_pagina = scrapear_pagina(url, es_primera)

        # En la primera página guardamos el total de anuncios del distrito
        if pagina == 1 and total_pagina > 0:
            total_anuncios_barrio = total_pagina

        # Si no se obtuvieron datos, fin para este distrito
        if len(datos_pagina) == 0:
            print(f"   Sin datos en página {pagina}. Fin del distrito {barrio}.")
            break

        # Acumular datos
        datos_scraped.extend(datos_pagina)
        anuncios_scrapeados_barrio += len(datos_pagina)
        print(f"   Acumulados hasta ahora en {barrio}: {anuncios_scrapeados_barrio}")

        # ── Guardado incremental del CSV ──────────────────────
        # Se guarda después de cada página: si el proceso se interrumpe,
        # los datos ya scrapeados no se pierden
        if len(datos_scraped) > 0:
            df_parcial = pd.DataFrame(datos_scraped)
            df_parcial.to_csv(output_file, index=False, encoding='utf-8-sig')
            print(f"   CSV actualizado: {len(datos_scraped)} registros totales guardados.")

        # ── Corte de prueba ──────────────────────────────────
        # IMPORTANTE: cambiar LIMITE_PAGINAS al inicio del script
        # para el scraping completo (poner 999 o similar)
        if pagina >= LIMITE_PAGINAS:
            print(f"   Corte de prueba alcanzado ({pagina} páginas). "
                  f"Subir LIMITE_PAGINAS para el scraping completo.")
            break

        # ── Comprobar si ya hemos llegado al total del distrito ──
        if total_anuncios_barrio > 0 and anuncios_scrapeados_barrio >= total_anuncios_barrio:
            print(f"   ¡Completado! {anuncios_scrapeados_barrio} / {total_anuncios_barrio} "
                  f"anuncios scrapeados en {barrio}")
            break

        # ── Pausa entre páginas ───────────────────────────────
        # Con la nueva estrategia de sesión nueva por página, esta pausa
        # ya no es crítica para evitar detección, pero la mantenemos
        # para no sobrecargar el servidor de Fotocasa
        pausa = rd.uniform(15, 25)
        print(f"   Esperando {pausa:.0f}s antes de la siguiente página...")
        time.sleep(pausa)

        pagina += 1

    # ── Pausa entre distritos ────────────────────────────────
    # (15s para pruebas; subir a 120s para el scraping completo multi-distrito)
    print(f"\n   Fin del distrito {barrio}. Esperando antes del siguiente...")
    time.sleep(15)


# ── RESUMEN FINAL ────────────────────────────────────────────
if len(datos_scraped) > 0:
    df = pd.DataFrame(datos_scraped)

    print("\n" + "="*60)
    print(f"Scraping completado. Total de anuncios: {len(df)}")
    print("="*60)

    print("\nColumnas del dataset:")
    print(df.columns.tolist())

    print("\nPrimeros registros:")
    print(df.head(10).to_string())

    # Guardar versión final (ya se habrá guardado incrementalmente, esto es redundante pero seguro)
    df.to_csv(output_file, index=False, encoding='utf-8-sig')  # utf-8-sig para Excel
    print(f"\nDatos guardados en: {output_file}")

    # Estadísticas rápidas
    print("\nResumen por barrio:")
    print(df['barrio'].value_counts().head(20))

    print("\nRango de precios encontrados:")
    print(df['precio'].value_counts().head(10))

else:
    print("\nNo se obtuvieron datos. Verifica que:")
    print("   1. La URL sea accesible")
    print("   2. El sitio no te esté bloqueando")
    print("   3. Los selectores CSS sean correctos")
    print("   4. Las esperas sean suficientes para la carga de la página")


---
# 3. Sistema de extracción de datos turísticos de Airbnb

## 3.1. Motivación y objetivos

Completada la base de datos de Idealista, se identificó la necesidad de incorporar información del **mercado de alquiler turístico** para estudiar la posible influencia de las viviendas de uso turístico sobre el mercado inmobiliario tradicional. Se seleccionó **Airbnb** por su relevancia internacional y la elevada concentración de alojamientos en las principales ciudades españolas.

Variables objetivo por anuncio:

- Nombre del alojamiento.
- Descripción del anuncio.
- Precio ofertado.
- Enlace directo al anuncio.
- **Fecha de disponibilidad consultada** (check-in).

A diferencia de Idealista, donde interesaba una fotografía estática del mercado, en Airbnb era necesario analizar la **evolución temporal** de la oferta. Por ello el sistema se diseñó desde el inicio para construir una **base de datos longitudinal**.

## 3.2. Primera versión: 3 ciudades con bucle diario de 365 días

**Script original:** `webscraping airbnb-3 ciudades.py`

El desarrollo reutilizó la experiencia adquirida con Idealista: misma configuración anti-detección del navegador, modificación de indicadores de Selenium, User-Agent realista y modo incógnito.

Las pruebas se realizaron sobre **Madrid, Barcelona y Valencia**. Para cada ciudad se generan automáticamente las URL de búsqueda incorporando parámetros de **check-in y check-out** para estancias de una noche, calculados con la librería `datetime`:

```
https://www.airbnb.com/s/{zona}--{Ciudad}--Spain/homes?checkin=YYYY-MM-DD&checkout=YYYY-MM-DD
```

### Gestión de contenido dinámico (WebDriverWait)

Gran parte del contenido de Airbnb se genera mediante **JavaScript después de la carga inicial**, por lo que el HTML inicial no garantiza la disponibilidad de la información. Para resolverlo se incorporaron **esperas explícitas** con `WebDriverWait`, que verifican la presencia efectiva de los contenedores de anuncios (`[data-testid='card-container']`) antes de extraer. Complementariamente, `time.sleep(5 + page)` introduce pausas que **aumentan progresivamente** con cada página, asegurando el renderizado completo y reduciendo la probabilidad de detección.

La paginación se realiza pulsando el botón *Siguiente* (`a[aria-label='Siguiente']`) hasta agotar los resultados.

### Limitación detectada

Consultar **cada uno de los 365 días** posteriores a la ejecución implicaba un volumen muy elevado de peticiones (365 días × 3 ciudades × N páginas) y tiempos de ejecución extremadamente prolongados, incrementando el riesgo de detección y dificultando la gestión de recursos.

In [ ]:
# Web scraping de Airbnb

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
from datetime import datetime, timedelta

# Configuración para Windows - IMPORTANTE: argumentos para evitar que Chrome se cierre
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-web-resources")
options.add_argument("--no-first-run")
options.add_argument("--no-default-browser-check")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
# Agregar User-Agent más realista para evitar bloqueos
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
# Desactivar automation indicators
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# Agregar preferencias para simular navegador real
options.add_experimental_option("prefs", {
    "profile.password_manager_enabled": False,
    "credentials_enable_service": False,
    "profile.default_content_setting_values.notifications": 2,
    "profile.managed_default_content_settings.images": 2,  # Deshabilitar imágenes
    "profile.default_content_settings.popups": 0,
    "profile.default_content_settings.plugins": 2,
})
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins")
options.add_argument("--disable-images")  # Para acelerar carga
options.add_argument("--disable-background-timer-throttling")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-features=VizDisplayCompositor")
# Crear perfil temporal para evitar detección
options.add_argument("--incognito")
options.add_argument("--disable-web-security")
options.add_argument("--allow-running-insecure-content")

service = Service(ChromeDriverManager().install())

# Ciudades a scrapear
ciudades = ["madrid centro", "barcelona centro", "valencia centro"]
nombres_ciudades = ["madrid", "barcelona", "valencia"]

# Iterar sobre cada ciudad
for idx, zona in enumerate(ciudades):
    nombre_ciudad = nombres_ciudades[idx]

    # Lista para almacenar todos los anuncios de los 365 días
    todos_los_anuncios = []

    # Iterar 365 veces (una para cada día del año)
    for dia in range(365):
        # Calcular las fechas para cada día
        hoy = datetime.now().date() + timedelta(days=dia)
        manana = hoy + timedelta(days=1)
        checkin = hoy.strftime('%Y-%m-%d')
        checkout = manana.strftime('%Y-%m-%d')

        # Construir la URL de búsqueda en Airbnb
        query = zona.replace(" ", "-")
        ciudad_capitalized = zona.split()[0].capitalize()
        url = f"https://www.airbnb.com/s/{query}--{ciudad_capitalized}--Spain/homes?checkin={checkin}&checkout={checkout}"

        try:
            driver = webdriver.Chrome(service=service, options=options)
            print(f"\n{'='*60}")
            print(f"Iniciando scraping para: {zona.upper()} - Día {dia + 1}/365 (Check-in: {checkin})")
            print(f"{'='*60}")
            print("Conexión con Chrome establecida correctamente")

            # Ejecutar JavaScript para eliminar indicadores de Selenium
            driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
            driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")

            # Navegar a la URL
            driver.get(url)
            print(f"Navegando a: {url}")

            anuncios = []
            page = 1
            max_pages = 50  # Límite máximo de páginas para evitar loops infinitos

            while page <= max_pages:
                print(f"Procesando página {page}")

                # Navegar a la URL (solo en la primera página, luego se hace clic en siguiente)
                if page == 1:
                    driver.get(url)
                    print(f"Navegando a: {url}")

                # Aceptar cookies si aparecen (solo en la primera página)
                if page == 1:
                    try:
                        cookie_button = WebDriverWait(driver, 10).until(
                            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept') or contains(text(), 'Aceptar') or contains(text(), 'Agree')]"))
                        )
                        cookie_button.click()
                        print("Cookies aceptadas")
                        time.sleep(2)
                    except:
                        print("No se encontró botón de cookies o ya aceptadas")

                # Esperar a que cargue la página (esperar por los contenedores de anuncios)
                WebDriverWait(driver, 20).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "[data-testid='card-container']"))
                )
                time.sleep(5 + page)  # Espera adicional que aumenta con cada página para simular comportamiento humano

                # Encontrar los contenedores de los anuncios en la página actual
                cards = driver.find_elements(By.CSS_SELECTOR, "[data-testid='card-container']")

                print(f"Encontrados {len(cards)} anuncios en la página {page}")

                for card in cards:
                    try:
                        # Link al anuncio
                        link_element = card.find_element(By.CSS_SELECTOR, "a[href*='/rooms/']")
                        link = link_element.get_attribute("href")

                        # Nombre del anuncio
                        name_element = card.find_element(By.CSS_SELECTOR, "[data-testid='listing-card-title']")
                        nombre = name_element.text

                        # Descripción (puede no estar siempre presente)
                        try:
                            desc_element = card.find_element(By.CSS_SELECTOR, "[data-testid='listing-card-subtitle']")
                            descripcion = desc_element.text
                        except:
                            descripcion = ""

                        # Precio
                        try:
                            price_element = card.find_element(By.CSS_SELECTOR, "[data-testid='price-availability-row'] span")
                            precio = price_element.text
                        except:
                            precio = ""

                        anuncios.append({
                            'link': link,
                            'nombre': nombre,
                            'descripcion': descripcion,
                            'precio': precio,
                            'fecha_checkin': checkin
                        })
                    except Exception as e:
                        print(f"Error extrayendo datos de un anuncio: {e}")
                        continue

                # Intentar ir a la siguiente página
                try:
                    next_button = WebDriverWait(driver, 10).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, "a[aria-label='Siguiente']"))
                    )
                    next_button.click()
                    page += 1
                    time.sleep(5 + page)  # Esperar a que cargue la nueva página, aumentando con cada página
                except:
                    print("No hay más páginas disponibles")
                    break

            print(f"Anuncios extraídos para {zona} (Día {dia + 1}): {len(anuncios)}")

            # Agregar los anuncios del día a la lista general
            todos_los_anuncios.extend(anuncios)

            # Cerrar el navegador
            driver.quit()
            print(f"Scraping completado para {zona.upper()} - Día {dia + 1}\n")

        except Exception as e:
            print(f"Error en {zona} - Día {dia + 1}: {e}")
            if 'driver' in locals():
                driver.quit()

    # Al finalizar los 365 días para esta ciudad, guardar todos los datos en un único archivo
    print(f"\n{'='*60}")
    print(f"Guardando datos finales para {nombre_ciudad.upper()}")
    print(f"Total de anuncios recolectados: {len(todos_los_anuncios)}")
    print(f"{'='*60}\n")

    # Crear DataFrame con todos los anuncios de los 365 días
    df_final = pd.DataFrame(todos_los_anuncios)

    # Mostrar el DataFrame
    print(df_final.head(10))

    # Guardar el DataFrame en un CSV con el nombre de la ciudad
    fecha_hoy = datetime.now().date().strftime('%Y-%m-%d')
    ruta_csv = f'D:\\Pontia\\Master analisis de datos\\Pruebas\\datos_airbnb_{nombre_ciudad}_365dias_{fecha_hoy}.csv'
    df_final.to_csv(ruta_csv, index=False)
    print(f"DataFrame guardado en '{ruta_csv}'")

## 3.3. Optimización mediante muestreo temporal semanal

**Script original:** `webscraping airbnb-3 ciudades 365 7 en 7.py`

Para mejorar la eficiencia se adoptó una estrategia de **muestreo temporal por intervalos de 7 días**: en lugar de consultar cada fecha, el bucle avanza `dia += 7`, generando ~52 consultas por ciudad en lugar de 365.

Ventajas de esta aproximación:

- Reducción significativa del tiempo total de ejecución (~86% menos consultas).
- Disminución del volumen de peticiones enviadas a la plataforma.
- Menor probabilidad de detección por tráfico automatizado.
- Conservación de una **cobertura temporal suficiente** para el análisis de estacionalidad.

Cada alojamiento se almacena junto con su `fecha_checkin`, lo que permite reconstruir la evolución temporal de la oferta. Al finalizar cada ciudad, todos los registros se exportan a un único CSV identificado con el nombre de la ciudad y la fecha de ejecución.

Esta estrategia representó un **equilibrio adecuado entre profundidad temporal y viabilidad operativa**.

In [ ]:
# Web scraping de Airbnb

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
from datetime import datetime, timedelta

# Configuración para Windows - IMPORTANTE: argumentos para evitar que Chrome se cierre
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-web-resources")
options.add_argument("--no-first-run")
options.add_argument("--no-default-browser-check")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
# Agregar User-Agent más realista para evitar bloqueos
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
# Desactivar automation indicators
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# Agregar preferencias para simular navegador real
options.add_experimental_option("prefs", {
    "profile.password_manager_enabled": False,
    "credentials_enable_service": False,
    "profile.default_content_setting_values.notifications": 2,
    "profile.managed_default_content_settings.images": 2,  # Deshabilitar imágenes
    "profile.default_content_settings.popups": 0,
    "profile.default_content_settings.plugins": 2,
})
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins")
options.add_argument("--disable-images")  # Para acelerar carga
options.add_argument("--disable-background-timer-throttling")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-features=VizDisplayCompositor")
# Crear perfil temporal para evitar detección
options.add_argument("--incognito")
options.add_argument("--disable-web-security")
options.add_argument("--allow-running-insecure-content")

service = Service(ChromeDriverManager().install())

# Ciudades a scrapear
ciudades = ["madrid centro", "barcelona centro", "valencia centro"]
nombres_ciudades = ["madrid", "barcelona", "valencia"]

# Iterar sobre cada ciudad
for idx, zona in enumerate(ciudades):
    nombre_ciudad = nombres_ciudades[idx]

    # Lista para almacenar todos los anuncios de los 365 días
    todos_los_anuncios = []

    # Iterar de 7 en 7 días hasta que pasen al menos 365 días
    dia = 0
    numero_semana = 1
    while dia < 365:
        # Calcular las fechas para cada día (en incrementos de 7 días)
        hoy = datetime.now().date() + timedelta(days=dia)
        manana = hoy + timedelta(days=1)
        checkin = hoy.strftime('%Y-%m-%d')
        checkout = manana.strftime('%Y-%m-%d')

        # Construir la URL de búsqueda en Airbnb
        query = zona.replace(" ", "-")
        ciudad_capitalized = zona.split()[0].capitalize()
        url = f"https://www.airbnb.com/s/{query}--{ciudad_capitalized}--Spain/homes?checkin={checkin}&checkout={checkout}"

        try:
            driver = webdriver.Chrome(service=service, options=options)
            print(f"\n{'='*60}")
            print(f"Iniciando scraping para: {zona.upper()} - Semana {numero_semana} (Día {dia + 1}/365+) (Check-in: {checkin})")
            print(f"{'='*60}")
            print("Conexión con Chrome establecida correctamente")

            # Ejecutar JavaScript para eliminar indicadores de Selenium
            driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
            driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")

            # Navegar a la URL
            driver.get(url)
            print(f"Navegando a: {url}")

            anuncios = []
            page = 1
            max_pages = 50  # Límite máximo de páginas para evitar loops infinitos

            while page <= max_pages:
                print(f"Procesando página {page}")

                # Navegar a la URL (solo en la primera página, luego se hace clic en siguiente)
                if page == 1:
                    driver.get(url)
                    print(f"Navegando a: {url}")

                # Aceptar cookies si aparecen (solo en la primera página)
                if page == 1:
                    try:
                        cookie_button = WebDriverWait(driver, 10).until(
                            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept') or contains(text(), 'Aceptar') or contains(text(), 'Agree')]"))
                        )
                        cookie_button.click()
                        print("Cookies aceptadas")
                        time.sleep(2)
                    except:
                        print("No se encontró botón de cookies o ya aceptadas")

                # Esperar a que cargue la página (esperar por los contenedores de anuncios)
                WebDriverWait(driver, 20).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "[data-testid='card-container']"))
                )
                time.sleep(5 + page)  # Espera adicional que aumenta con cada página para simular comportamiento humano

                # Encontrar los contenedores de los anuncios en la página actual
                cards = driver.find_elements(By.CSS_SELECTOR, "[data-testid='card-container']")

                print(f"Encontrados {len(cards)} anuncios en la página {page}")

                for card in cards:
                    try:
                        # Link al anuncio
                        link_element = card.find_element(By.CSS_SELECTOR, "a[href*='/rooms/']")
                        link = link_element.get_attribute("href")

                        # Nombre del anuncio
                        name_element = card.find_element(By.CSS_SELECTOR, "[data-testid='listing-card-title']")
                        nombre = name_element.text

                        # Descripción (puede no estar siempre presente)
                        try:
                            desc_element = card.find_element(By.CSS_SELECTOR, "[data-testid='listing-card-subtitle']")
                            descripcion = desc_element.text
                        except:
                            descripcion = ""

                        # Precio
                        try:
                            price_element = card.find_element(By.CSS_SELECTOR, "[data-testid='price-availability-row'] span")
                            precio = price_element.text
                        except:
                            precio = ""

                        anuncios.append({
                            'link': link,
                            'nombre': nombre,
                            'descripcion': descripcion,
                            'precio': precio,
                            'fecha_checkin': checkin
                        })
                    except Exception as e:
                        print(f"Error extrayendo datos de un anuncio: {e}")
                        continue

                # Intentar ir a la siguiente página
                try:
                    next_button = WebDriverWait(driver, 10).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, "a[aria-label='Siguiente']"))
                    )
                    next_button.click()
                    page += 1
                    time.sleep(5 + page)  # Esperar a que cargue la nueva página, aumentando con cada página
                except:
                    print("No hay más páginas disponibles")
                    break

            print(f"Anuncios extraídos para {zona} (Semana {numero_semana} - Día {dia + 1}): {len(anuncios)}")

            # Agregar los anuncios del día a la lista general
            todos_los_anuncios.extend(anuncios)

            # Cerrar el navegador
            driver.quit()
            print(f"Scraping completado para {zona.upper()} - Semana {numero_semana}\n")

            # Incrementar para la siguiente semana
            dia += 7
            numero_semana += 1

        except Exception as e:
            print(f"Error en {zona} - Semana {numero_semana}: {e}")
            if 'driver' in locals():
                driver.quit()
            # Incrementar incluso si hay error, para continuar con la siguiente semana
            dia += 7
            numero_semana += 1

    # Al finalizar los 365 días para esta ciudad, guardar todos los datos en un único archivo
    print(f"\n{'='*60}")
    print(f"Guardando datos finales para {nombre_ciudad.upper()}")
    print(f"Total de anuncios recolectados: {len(todos_los_anuncios)}")
    print(f"{'='*60}\n")

    # Crear DataFrame con todos los anuncios de los 365 días
    df_final = pd.DataFrame(todos_los_anuncios)

    # Mostrar el DataFrame
    print(df_final.head(10))

    # Guardar el DataFrame en un CSV con el nombre de la ciudad
    fecha_hoy = datetime.now().date().strftime('%Y-%m-%d')
    ruta_csv = f'D:\\Pontia\\Master analisis de datos\\Pruebas\\datos_airbnb_{nombre_ciudad}_365dias_de7en7_{fecha_hoy}.csv'
    df_final.to_csv(ruta_csv, index=False)
    print(f"DataFrame guardado en '{ruta_csv}'")

## 3.4. Versión final: análisis por barrios de Valencia y robustez de sesión

**Script original:** `Web scraping de Airbnb Valencia 4b.py`

La última evolución del sistema aumentó la **granularidad espacial** (de ciudad a **barrio**: Ciutat Vella, Benimaclet, Cabanyal-Canyamelar y Eixample) e incorporó mejoras de robustez para ejecuciones de muy larga duración:

1. **Pausas aleatorias humanizadas** — la función `human_sleep(min, max)` sustituye las esperas fijas por tiempos variables (`random.uniform`), evitando los patrones periódicos fácilmente detectables de las versiones anteriores.

2. **Reinicio periódico del navegador** — cada 50 días consultados se cierra y reabre Chrome (`restart_periodo = 50`), simulando varias sesiones de usuario independientes en lugar de un proceso continuo, y reduciendo el consumo de memoria.

3. **Recuperación ante errores** — si una iteración falla, el navegador se reinicia automáticamente (reinyectando las propiedades JavaScript anti-detección) y el bucle continúa con el día siguiente, sin perder lo ya recopilado.

4. **Limpieza de cookies** (`driver.delete_all_cookies()`) antes de cada consulta y selector de cookies ampliado con más variantes de texto del botón de aceptación.

5. **Columna `barrio`** añadida al esquema de datos, habilitando el análisis espacial intraurbano.

Todos los barrios se consolidan en un **único DataFrame final** exportado a CSV, listo para el análisis posterior.

In [ ]:
# Web scraping de Airbnb

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import random
import pandas as pd
from datetime import datetime, timedelta

# Función para dormir un tiempo variable bajo comportamiento humano
def human_sleep(min_seconds=1.5, max_seconds=3.0):
    time.sleep(random.uniform(min_seconds, max_seconds))

# Configuración para Windows - IMPORTANTE: argumentos para evitar que Chrome se cierre
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-web-resources")
options.add_argument("--no-first-run")
options.add_argument("--no-default-browser-check")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
# Agregar User-Agent más realista para evitar bloqueos
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
# Desactivar automation indicators
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# Agregar preferencias para simular navegador real
options.add_experimental_option("prefs", {
    "profile.password_manager_enabled": False,
    "credentials_enable_service": False,
    "profile.default_content_setting_values.notifications": 2,
    "profile.managed_default_content_settings.images": 2,  # Deshabilitar imágenes
    "profile.default_content_settings.popups": 0,
    "profile.default_content_settings.plugins": 2,
})
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins")
options.add_argument("--disable-images")  # Para acelerar carga
options.add_argument("--disable-background-timer-throttling")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-features=VizDisplayCompositor")
# Crear perfil temporal para evitar detección
options.add_argument("--incognito")
options.add_argument("--disable-web-security")
options.add_argument("--allow-running-insecure-content")

service = Service(ChromeDriverManager().install())

# Barrios de Valencia para scrapear
barrios_valencia = [
    "ciutat vella",
    "benimaclet",
    "cabanyal-canyamelar",
    "eixample"
]
ciudad = "valencia"

todos_los_anuncios = []

# Iterar sobre cada barrio de Valencia
for barrio in barrios_valencia:
    nombre_barrio = barrio.replace('-', ' ').title()

    # Lista para almacenar todos los anuncios del barrio
    anuncios_barrio = []

    dias_a_scrapear = 365
    # reinicia el navegador cada 50 días para reducir la probabilidad de detección
    # y simular varias sesiones de usuario en lugar de un proceso continuo
    restart_periodo = 50

    driver = webdriver.Chrome(service=service, options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
    driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")

    try:
        for dia in range(dias_a_scrapear):
            hoy = datetime.now().date() + timedelta(days=dia)
            manana = hoy + timedelta(days=1)
            checkin = hoy.strftime('%Y-%m-%d')
            checkout = manana.strftime('%Y-%m-%d')

            # Construir la URL de búsqueda en Airbnb para el barrio de Valencia
            query = barrio.replace(' ', '-').replace('--', '-')
            url = f"https://www.airbnb.com/s/{query}--{ciudad.capitalize()}--Spain/homes?checkin={checkin}&checkout={checkout}"

            if dia > 0 and dia % restart_periodo == 0:
                print(f"\nReiniciando navegador después de {dia} días para reducir detección")
                driver.quit()
                human_sleep(2.0, 4.0)
                driver = webdriver.Chrome(service=service, options=options)
                driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
                driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
                driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")

            try:
                print(f"\n{'='*60}")
                print(f"Iniciando scraping para: {nombre_barrio.upper()} - Día {dia + 1}/{dias_a_scrapear} (Check-in: {checkin})")
                print(f"{'='*60}")
                print("Conexión con Chrome lista")

                driver.delete_all_cookies()
                driver.get(url)
                print(f"Navegando a: {url}")

                anuncios = []
                page = 1
                max_pages = 20  # Límite máximo de páginas para evitar loops infinitos

                while page <= max_pages:
                    print(f"Procesando página {page}")

                    if page == 1:
                        try:
                            cookie_button = WebDriverWait(driver, 3).until(
                                EC.element_to_be_clickable((By.XPATH,
                                    "//button[contains(text(), 'Accept') or contains(text(), 'Aceptar') or contains(text(), 'Agree') or contains(text(), 'Continuar') or contains(text(), 'Aceptar todo')]"
                                ))
                            )
                            cookie_button.click()
                            print("Cookies aceptadas")
                            human_sleep(1.5, 2.5)
                        except:
                            print("No se encontró botón de cookies o ya aceptadas")

                    WebDriverWait(driver, 15).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "[data-testid='card-container']"))
                    )
                    human_sleep(2.0, 3.5)

                    cards = driver.find_elements(By.CSS_SELECTOR, "[data-testid='card-container']")

                    print(f"Encontrados {len(cards)} anuncios en la página {page}")

                    for card in cards:
                        try:
                            link_element = card.find_element(By.CSS_SELECTOR, "a[href*='/rooms/']")
                            link = link_element.get_attribute("href")

                            name_element = card.find_element(By.CSS_SELECTOR, "[data-testid='listing-card-title']")
                            nombre = name_element.text

                            try:
                                desc_element = card.find_element(By.CSS_SELECTOR, "[data-testid='listing-card-subtitle']")
                                descripcion = desc_element.text
                            except:
                                descripcion = ""

                            try:
                                price_element = card.find_element(By.CSS_SELECTOR, "[data-testid='price-availability-row'] span")
                                precio = price_element.text
                            except:
                                precio = ""

                            anuncios.append({
                                'barrio': nombre_barrio,
                                'link': link,
                                'nombre': nombre,
                                'descripcion': descripcion,
                                'precio': precio,
                                'fecha_checkin': checkin
                            })
                        except Exception as e:
                            print(f"Error extrayendo datos de un anuncio: {e}")
                            continue

                    try:
                        next_button = WebDriverWait(driver, 10).until(
                            EC.element_to_be_clickable((By.CSS_SELECTOR, "a[aria-label='Siguiente'], a[aria-label='Next'], a[aria-label='Siguiente página']"))
                        )
                        next_button.click()
                        page += 1
                        human_sleep(2.0, 3.5)
                    except:
                        print("No hay más páginas disponibles")
                        break

                print(f"Anuncios extraídos para {nombre_barrio} - Día {dia + 1}: {len(anuncios)}")
                anuncios_barrio.extend(anuncios)
                human_sleep(2.0, 3.5)

            except Exception as e:
                print(f"Error en {nombre_barrio} - Día {dia + 1}: {e}")
                try:
                    driver.quit()
                except:
                    pass
                driver = webdriver.Chrome(service=service, options=options)
                driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
                driver.execute_script("Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]})")
                driver.execute_script("Object.defineProperty(navigator, 'languages', {get: () => ['es-ES', 'es']})")
                continue
    finally:
        try:
            driver.quit()
        except:
            pass

    todos_los_anuncios.extend(anuncios_barrio)

    print(f"\n{'='*60}")
    print(f"Fin de barrio: {nombre_barrio.upper()}")
    print(f"Total de anuncios recolectados en este barrio: {len(anuncios_barrio)}")
    print(f"{'='*60}\n")

# Guardar todos los barrios en un único archivo después de completar el scraping
print(f"\n{'='*60}")
print("Guardando datos finales de todos los barrios")
print(f"Total de anuncios recolectados: {len(todos_los_anuncios)}")
print(f"{'='*60}\n")

df_final = pd.DataFrame(todos_los_anuncios)

print(df_final.head(10))

fecha_hoy = datetime.now().date().strftime('%Y-%m-%d')
ruta_csv = f"D:\\Pontia\\Master analisis de datos\\Pruebas\\datos_airbnb_valencia_365dias_{fecha_hoy}.csv"
df_final.to_csv(ruta_csv, index=False)
print(f"DataFrame guardado en '{ruta_csv}'")

---
# 4. Conclusiones generales del proceso de web scraping

El desarrollo de los sistemas de extracción constituyó una etapa fundamental del trabajo, al permitir construir las bases de datos necesarias para el análisis del mercado residencial y turístico. La experiencia puso de manifiesto la creciente complejidad de la extracción automatizada en plataformas con mecanismos avanzados de protección.

**En el caso de Idealista**, la evolución desde peticiones HTTP convencionales hacia una arquitectura basada en Selenium permitió superar las limitaciones de acceso, y la migración de Google Colab a un entorno local proporcionó la estabilidad operativa necesaria para implementar técnicas avanzadas de automatización.

**En el caso de Airbnb**, los desafíos adicionales derivaron de la naturaleza dinámica de la plataforma y de la necesidad de incorporar la dimensión temporal. La combinación de Selenium, esperas explícitas (`WebDriverWait`) y estrategias de temporización permitió construir una base de datos longitudinal con cobertura de hasta un año en varias ciudades.

Elementos metodológicos determinantes para la obtención de datos válidos y consistentes:

- Configuración avanzada del navegador y ocultación de indicadores de automatización.
- Gestión de contenido dinámico mediante esperas explícitas.
- Pausas temporizadas (fijas primero, aleatorias después) y simulación de comportamiento humano.
- Adaptación continua a los mecanismos de protección de cada plataforma.

Como resultado se obtuvieron **dos conjuntos de datos complementarios**: la base de viviendas en venta de Idealista (estructura del mercado residencial) y la base longitudinal de alojamientos de Airbnb (evolución de la oferta turística), cuya integración proporciona el soporte empírico para los análisis estadísticos y modelos predictivos de las fases posteriores del trabajo.

---
# 5. Combinar CSVs Idealista y Fotocasa por distritos

Vamos a llevar a cabo la unificación de las extracciones de ambos portales inmobiliarios, hechas en grupos de 2-3 distritos para facilitar la gestión del código y la información, en 1 solo csv para cada portal y poder así llevar a cabo el análisis conjunto de los datos de cada portal en las 3 ciudades.


## 5.1. Configuración: librerías y rutas de entrada/salida

Se importan las librerías necesarias (`glob` para localizar todos los archivos CSV de una carpeta, `os` para gestión de rutas y `pandas` para la manipulación tabular) y se monta Google Drive, donde se almacenan tanto los CSV generados por los scrapers de Idealista y Fotocasa como los archivos combinados de salida.

In [ ]:
import glob
import os
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Se definen a continuación las rutas de las carpetas que contienen los CSV scrapeados de cada portal (uno por cada tanda de distritos) y las rutas donde se guardarán los dos CSV combinados finales, uno por portal.

In [ ]:
# Carpeta con los CSVs de Fotocasa (todos los archivos .csv dentro)
RUTA_FOTOCASA = r"/content/drive/MyDrive/Master Data Analysis/Proyecto Júpiter - HOSTING/Código PYTHON Proyecto & Csvs scrapeados/Código & Datos Idealista Fotocasa/Datos Fotocasa Limpios Analizados (Final)/Csv por separado/*.csv"

# Carpeta con los CSVs de Idealista
RUTA_IDEALISTA = r"/content/drive/MyDrive/Master Data Analysis/Proyecto Júpiter - HOSTING/Código PYTHON Proyecto & Csvs scrapeados/Código & Datos Idealista Fotocasa/Datos Idealista Limpios Analizados (Final)/Scraps para combinar/*.csv"

# Dónde guardar los CSV combinados finales
OUTPUT_FOTOCASA  = r"/content/drive/MyDrive/Master Data Analysis/Proyecto Júpiter - HOSTING/Código PYTHON Proyecto & Csvs scrapeados/Código & Datos Idealista Fotocasa/Datos Fotocasa Limpios Analizados (Final)/csv_combinado_datos_fotocasa/csv_fotocasa_combinado.csv"
OUTPUT_IDEALISTA = r"/content/drive/MyDrive/Master Data Analysis/Proyecto Júpiter - HOSTING/Código PYTHON Proyecto & Csvs scrapeados/Código & Datos Idealista Fotocasa/Datos Idealista Limpios Analizados (Final)/csv_combinado_datos_idealista/csv_idealista_combinado.csv"


## 5.2. Función auxiliar para detectar la ciudad

Como los CSV se generaron en tandas separadas por ciudad (el nombre del archivo incluye "Barcelona", "Madrid" o "Valencia"), se define una función que lee el nombre del archivo y asigna la ciudad correspondiente como una nueva columna, para poder identificar la procedencia de cada anuncio una vez combinados todos los distritos en un único CSV.

In [ ]:
def detectar_ciudad(nombre_archivo):
    """Devuelve la ciudad según el nombre del archivo CSV."""
    nombre = nombre_archivo.lower()
    if "barcelona" in nombre:
        return "Barcelona"
    elif "madrid" in nombre:
        return "Madrid"
    elif "valencia" in nombre:
        return "Valencia"
    else:
        return "Desconocido"

## 5.3. Combinación de los CSV de Fotocasa

Se listan primero todos los archivos CSV encontrados en la carpeta de Fotocasa, para verificar que se recogen todas las tandas de scraping antes de combinarlas.

In [ ]:
archivos_fotocasa = glob.glob(RUTA_FOTOCASA)
print(f"Archivos Fotocasa encontrados: {len(archivos_fotocasa)}")
for f in archivos_fotocasa:
    print(f"  - {os.path.basename(f)}")

Se cargan uno a uno los CSV encontrados, añadiendo la columna `ciudad` mediante la función anterior, y se concatenan todos en un único DataFrame. A continuación se aplica `drop_duplicates()`, que elimina las filas en las que **todas** las columnas coinciden exactamente (mismo precio, barrio, nombre, link, metros...), lo que retira los duplicados exactos que se producen cuando el mismo anuncio aparece en más de una tanda de scraping. Esta es una limpieza básica en origen; la depuración más fina de duplicados y valores atípicos se aborda más adelante en el análisis exploratorio (EDA).

In [ ]:
dfs_fotocasa = []

for archivo in archivos_fotocasa:
    try:
        df = pd.read_csv(archivo, encoding="utf-8-sig")  # marcamos el utf-8-sig por si al achivo fue guardado desde Windows
        df["ciudad"] = detectar_ciudad(os.path.basename(archivo))
        dfs_fotocasa.append(df)
        print(f"Cargado: {os.path.basename(archivo)} — {len(df)} filas — Ciudad: {df['ciudad'].iloc[0]}")
    except Exception as e:
        print(f"Error leyendo {os.path.basename(archivo)}: {e}")

if dfs_fotocasa:
    df_fotocasa_final = pd.concat(dfs_fotocasa, ignore_index=True)
    # Eliminar duplicados exactos por si algún CSV se solapaba... pero veo que también nos ha quitado también los duplicados
    df_fotocasa_final.drop_duplicates(inplace=True)
    print(f"\nTotal filas Fotocasa (sin duplicados): {len(df_fotocasa_final)}")
    print(f"\nDistribución por ciudad:")
    print(df_fotocasa_final["ciudad"].value_counts())
    df_fotocasa_final.to_csv(OUTPUT_FOTOCASA, index=False, encoding="utf-8-sig")
    print(f"\n CSV Fotocasa guardado en: {OUTPUT_FOTOCASA}")
else:
    print(" No se cargó ningún archivo de Fotocasa. Revisa la ruta RUTA_FOTOCASA.")

## 5.4. Combinación de los CSV de Idealista

Se repite exactamente el mismo procedimiento con los CSV de Idealista: listado de archivos, carga con asignación de ciudad, concatenación y eliminación de duplicados exactos.

In [ ]:
archivos_idealista = glob.glob(RUTA_IDEALISTA)
print(f"Archivos Idealista encontrados: {len(archivos_idealista)}")
for f in archivos_idealista:
    print(f"  - {os.path.basename(f)}")

In [ ]:
dfs_idealista = []

for archivo in archivos_idealista:
    try:
        df = pd.read_csv(archivo, encoding="utf-8-sig")
        df["ciudad"] = detectar_ciudad(os.path.basename(archivo))
        dfs_idealista.append(df)
        print(f"✔ Cargado: {os.path.basename(archivo)} — {len(df)} filas — Ciudad: {df['ciudad'].iloc[0]}")
    except Exception as e:
        print(f"✘ Error leyendo {os.path.basename(archivo)}: {e}")

if dfs_idealista:
    df_idealista_final = pd.concat(dfs_idealista, ignore_index=True)
    df_idealista_final.drop_duplicates(inplace=True)
    print(f"\nTotal filas Idealista (sin duplicados): {len(df_idealista_final)}")
    print(f"\nDistribución por ciudad:")
    print(df_idealista_final["ciudad"].value_counts())
    df_idealista_final.to_csv(OUTPUT_IDEALISTA, index=False, encoding="utf-8-sig")
    print(f"\n CSV Idealista guardado en: {OUTPUT_IDEALISTA}")
else:
    print(" No se cargó ningún archivo de Idealista. Revisa la ruta RUTA_IDEALISTA.")

## 5.5. Resumen final y verificación

Se imprime un resumen del volumen total de anuncios combinados por portal y su distribución por ciudad, y se inspecciona una muestra aleatoria de cada dataset combinado para comprobar visualmente que la unión se ha realizado correctamente.

In [ ]:
print("="*50)
print("RESUMEN FINAL")
print("="*50)

if dfs_fotocasa:
    print(f"\nFotocasa: {len(df_fotocasa_final):,} anuncios")
    print(df_fotocasa_final["ciudad"].value_counts().to_string())

if dfs_idealista:
    print(f"\nIdealista: {len(df_idealista_final):,} anuncios")
    print(df_idealista_final["ciudad"].value_counts().to_string())

In [ ]:
df_fotocasa_final.sample(10)

In [ ]:
df_idealista_final.sample(10)

## 5.6. Corrección de ciudades no identificadas

Al revisar la distribución por ciudad se detectó que la función `detectar_ciudad()` había dejado un grupo de registros de Fotocasa etiquetados como "Desconocido", al no reconocer el nombre de la ciudad en el nombre del archivo correspondiente.

In [ ]:
df_fotocasa_final[df_fotocasa_final['ciudad'] == 'Desconocido']

Para intentar recuperar esos registros, se define una segunda función que busca el nombre de la ciudad directamente en el texto de la columna `barrio` de cada anuncio, y se aplica únicamente sobre las filas que habían quedado como "Desconocido".

In [ ]:
def detectar_ciudad_desde_barrio(barrio):
    if pd.isna(barrio):
        return "Desconocido"
    barrio = str(barrio).lower()
    if "barcelona" in barrio:
        return "Barcelona"
    elif "madrid" in barrio:
        return "Madrid"
    elif "valencia" in barrio:
        return "Valencia"
    else:
        return "Desconocido"

# Aplicar solo a las filas que quedaron como 'Desconocido'
mask = df_fotocasa_final["ciudad"] == "Desconocido"
df_fotocasa_final.loc[mask, "ciudad"] = (
    df_fotocasa_final.loc[mask, "barrio"].apply(detectar_ciudad_desde_barrio)
)

print(f"Filas corregidas: {mask.sum()}")
print(f"\nDistribución por ciudad tras corrección:")
print(df_fotocasa_final["ciudad"].value_counts())


In [ ]:
df_desconocido['barrio'].unique()

Tras la corrección, los registros que persisten como "Desconocido" (174 en total) corresponden a un límite real del proceso de scraping y no a un fallo de esta función de reasignación: son anuncios de localidades del área metropolitana de Barcelona —Hospitalet, Sabadell, Terrassa, entre otras— que pertenecen a la provincia pero no al municipio de Barcelona capital, y que quedan pendientes de una revisión y reclasificación manual posterior.

Por último, se guarda la versión corregida del CSV combinado de Fotocasa.

In [ ]:
OUTPUT_FOTOCASA_2  = r"/content/drive/MyDrive/Master Data Analysis/Proyecto Júpiter - HOSTING/Código PYTHON Proyecto & Csvs scrapeados/Código & Datos Idealista Fotocasa/Datos Fotocasa Limpios Analizados (Final)/csv_combinado_datos_fotocasa/csv_fotocasa_combinado_2.csv"
# Guardar el CSV corregido
df_fotocasa_final.to_csv(OUTPUT_FOTOCASA_2, index=False, encoding="utf-8-sig")
print(f"\n CSV actualizado guardado.")